In [25]:
# ============================================================
# 041_weekly_events_digest
# ============================================================
#
# Overview
# ----------------
# Weekly aggregation notebook that queries the past-week Events from Notion,
# deduplicates and filters noise, clusters events into interpretable themes,
# ranks themes by impact, renders a Markdown digest, and exports artifacts.
# Optionally calls an LLM (gpt-4o-mini) to generate theme titles/summaries/signals.
# Default timezone: Asia/Tokyo.
#
# Inputs / Outputs
# ----------------
# Inputs:
#   - NOTION_EVENTS_DB_ID: Notion *data source* id containing all Events (queried via /v1/data_sources/{id}/query)
#   - NOTION_WEEKLY_DIGESTS_DS_ID: Notion *data source* id for Weekly Digest pages (optional writeback)
#   - NOTION_TOKEN: Notion API token
#   - OPENAI_API_KEY: Optional (enables LLM theme interpretation)
#   - Time window: week_start–week_end computed in JST (typically last 7 days)
#   - Optional noise filters: MIN_CONFIDENCE_THRESHOLD, EXCLUDED_STATUSES, EXCLUDED_SOURCES
#
# Outputs:
#   - Local artifacts:
#       output/weekly_events_digest_<YYYY-MM-DD>.md
#       output/weekly_events_digest_<YYYY-MM-DD>.json
#   - Console summary + on-screen preview (top themes + digest excerpt)
#   - Optional Notion writeback:
#       Upsert a Weekly Digest page and append the digest content as blocks
#       (append-only for auditability; no destructive block deletion).
#
# Structure
# ----------------
# Cell 01: Imports, environment setup (.env), constants, logging, run_id
# Cell 02: Notion API wrappers (data_sources query) + property parsing helpers
# Cell 03: Time window computation (week_start–week_end in Asia/Tokyo)
# Cell 04: Fetch weekly Events from NOTION_EVENTS_DB_ID (data source query)
# Cell 05: Normalize records (schema-aligned) and validate required fields
# Cell 06: Deduplication + noise filtering (confidence/status/source)
# Cell 07: Theme clustering (centroid-updating greedy) + optional LLM interpretation
# Cell 08: Theme scoring + Top N selection + top events per theme
# Cell 09: Render weekly digest markdown (header + exec summary + themes + end summary)
# Cell 10: Notion writeback upsert to Weekly Digests (optional; append-only blocks)
# Cell 11: Export local artifacts + final run summary + on-screen preview
#
# Notes
# ----------------
# - Assumes Notion API ≥ 2025-09-03 and the "data_sources" paradigm (databases are containers).
# - Event schema expected (Events data source):
#     Name (title), Date (date), Detected At (date), Target (relation), Event Type (select),
#     Source URL (url), Source (select), Summary (rich_text), Confidence (number),
#     Dedup Key (rich_text), Status (select), plus optional operational fields.
# - Dedup: uses "Dedup Key" when present; otherwise creates a deterministic hash fallback.
# - Theme clustering relies primarily on target overlap + keyword overlap; event_type/source provide light boosts.
# - LLM interpretation is gated by OPENAI_API_KEY and USE_LLM_LABELING; outputs strict JSON (title/summary/why/signals).
# - Writeback is skipped if NOTION_WEEKLY_DIGESTS_DS_ID is not set.
# - The notebook is designed to be rerunnable (idempotent-ish): same week label will update the same digest page.


In [2]:
# ============================================================
# Cell 01 — Imports, environment setup, constants, logging, run_id
# ============================================================
# Overview:
#
# Inputs / Outputs:
#
# Notes:
#

# --- Mandatory env loading ---
from dotenv import load_dotenv
load_dotenv('env.txt')

# --- Runtime LLM configuration (given / assumed) ---
llm_provider = 'OpenAI'
llm_model = 'gpt-4o-mini'
llm_temperature = 0.0

# --- Imports ---
import os
import sys
import json
import logging
from datetime import datetime, timedelta
from pathlib import Path
from typing import Dict, List, Any, Optional, Set
from collections import defaultdict
import hashlib
import re

# Third-party
import requests
from zoneinfo import ZoneInfo

# --- Environment variables ---
NOTION_TOKEN = os.getenv('NOTION_TOKEN')
NOTION_EVENTS_DB_ID = os.getenv('NOTION_EVENTS_DB_ID')
NOTION_WEEKLY_DIGESTS_DS_ID = os.getenv('NOTION_WEEKLY_DIGESTS_DS_ID')
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

# --- Constants ---
TIMEZONE = ZoneInfo('Asia/Tokyo')
NOTION_API_VERSION = '2022-06-28'
NOTION_BASE_URL = 'https://api.notion.com/v1'

# Filtering thresholds
MIN_CONFIDENCE_THRESHOLD = 0.3
EXCLUDED_STATUSES = {'archived', 'deleted', 'spam'}
EXCLUDED_SOURCES = {'test', 'debug'}

# Clustering parameters
THEME_OVERLAP_THRESHOLD = 0.4
TOP_N_THEMES = 5
MAX_EVENTS_PER_THEME = 10

# LLM configuration
USE_LLM_LABELING = bool(OPENAI_API_KEY)

# Output directory
OUTPUT_DIR = Path('output')
OUTPUT_DIR.mkdir(exist_ok=True)

# --- Logging setup ---
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[
        logging.StreamHandler(sys.stdout)
    ]
)
logger = logging.getLogger(__name__)

# --- Run ID ---
RUN_ID = datetime.now(TIMEZONE).strftime('%Y%m%d_%H%M%S')
logger.info(f"Weekly Events Digest run started: {RUN_ID}")
logger.info(f"Timezone: {TIMEZONE}")
logger.info(f"LLM labeling enabled: {USE_LLM_LABELING}")

# --- Validation ---
if not NOTION_TOKEN:
    logger.error("NOTION_TOKEN not found in environment")
    raise ValueError("NOTION_TOKEN is required")

if not NOTION_EVENTS_DB_ID:
    logger.error("NOTION_EVENTS_DB_ID not found in environment")
    raise ValueError("NOTION_EVENTS_DB_ID is required")

if not NOTION_WEEKLY_DIGESTS_DS_ID:
    logger.warning("NOTION_WEEKLY_DIGESTS_DS_ID not set - Notion writeback will be skipped")

logger.info("Cell 01 complete: Environment configured")


2026-02-06 13:42:48,580 [INFO] Weekly Events Digest run started: 20260206_134248
2026-02-06 13:42:48,581 [INFO] Timezone: Asia/Tokyo
2026-02-06 13:42:48,581 [INFO] LLM labeling enabled: True
2026-02-06 13:42:48,582 [WARNING] NOTION_WEEKLY_DIGESTS_DS_ID not set - Notion writeback will be skipped
2026-02-06 13:42:48,583 [INFO] Cell 01 complete: Environment configured


In [3]:
# ============================================================
# Cell 02 — Notion API wrappers and schema helpers
# ============================================================
# Overview:
#
# Inputs / Outputs:
#
# Notes:
#

# --- Notion API headers ---
notion_headers = {
    'Authorization': f'Bearer {NOTION_TOKEN}',
    'Notion-Version': NOTION_API_VERSION,
    'Content-Type': 'application/json'
}

# --- Schema property extraction helpers ---

def get_property_value(properties: Dict, prop_name: str, prop_type: str, default: Any = None) -> Any:
    """
    Extract a property value from Notion page properties dict.
    
    Args:
        properties: Notion page properties dict
        prop_name: Property name
        prop_type: Expected type (title, rich_text, number, date, select, multi_select, checkbox, url, relation)
        default: Default value if not found or empty
    
    Returns:
        Extracted value or default
    """
    prop = properties.get(prop_name, {})
    
    if prop_type == 'title':
        title_list = prop.get('title', [])
        if title_list:
            return ''.join([t.get('plain_text', '') for t in title_list])
        return default
    
    elif prop_type == 'rich_text':
        text_list = prop.get('rich_text', [])
        if text_list:
            return ''.join([t.get('plain_text', '') for t in text_list])
        return default
    
    elif prop_type == 'number':
        return prop.get('number', default)
    
    elif prop_type == 'date':
        date_obj = prop.get('date')
        if date_obj:
            return date_obj.get('start', default)
        return default
    
    elif prop_type == 'select':
        select_obj = prop.get('select')
        if select_obj:
            return select_obj.get('name', default)
        return default
    
    elif prop_type == 'multi_select':
        multi = prop.get('multi_select', [])
        return [item.get('name') for item in multi if item.get('name')]
    
    elif prop_type == 'checkbox':
        return prop.get('checkbox', default)
    
    elif prop_type == 'url':
        return prop.get('url', default)
    
    elif prop_type == 'relation':
        relations = prop.get('relation', [])
        return [r.get('id') for r in relations if r.get('id')]
    
    return default


def build_property_value(prop_type: str, value: Any) -> Dict:
    """
    Build a Notion property value object for page creation/update.
    
    Args:
        prop_type: Property type
        value: Value to set
    
    Returns:
        Notion property object
    """
    if prop_type == 'title':
        return {'title': [{'text': {'content': str(value)}}]}
    
    elif prop_type == 'rich_text':
        return {'rich_text': [{'text': {'content': str(value)}}]}
    
    elif prop_type == 'number':
        return {'number': float(value) if value is not None else None}
    
    elif prop_type == 'date':
        return {'date': {'start': str(value)} if value else None}
    
    elif prop_type == 'select':
        return {'select': {'name': str(value)} if value else None}
    
    elif prop_type == 'multi_select':
        if isinstance(value, list):
            return {'multi_select': [{'name': str(v)} for v in value]}
        return {'multi_select': []}
    
    elif prop_type == 'checkbox':
        return {'checkbox': bool(value)}
    
    elif prop_type == 'url':
        return {'url': str(value) if value else None}
    
    return {}


# --- Notion API query wrapper ---

def query_database(database_id: str, filter_obj: Optional[Dict] = None, sorts: Optional[List[Dict]] = None, page_size: int = 100) -> List[Dict]:
    """
    Query a Notion database and return all results (handles pagination).
    
    Args:
        database_id: Notion database ID
        filter_obj: Optional filter object
        sorts: Optional sorts list
        page_size: Page size (max 100)
    
    Returns:
        List of page objects
    """
    url = f'{NOTION_BASE_URL}/databases/{database_id}/query'
    all_results = []
    has_more = True
    start_cursor = None
    
    while has_more:
        payload = {'page_size': page_size}
        if filter_obj:
            payload['filter'] = filter_obj
        if sorts:
            payload['sorts'] = sorts
        if start_cursor:
            payload['start_cursor'] = start_cursor
        
        response = requests.post(url, headers=notion_headers, json=payload)
        
        if response.status_code != 200:
            logger.error(f"Query database failed: {response.status_code} - {response.text}")
            raise Exception(f"Notion API query failed: {response.status_code}")
        
        data = response.json()
        all_results.extend(data.get('results', []))
        has_more = data.get('has_more', False)
        start_cursor = data.get('next_cursor')
    
    return all_results


# --- Notion page creation ---

def create_page(database_id: str, properties: Dict, children: Optional[List[Dict]] = None) -> Dict:
    """
    Create a new page in a Notion database.
    
    Args:
        database_id: Parent database ID
        properties: Page properties dict
        children: Optional list of block children
    
    Returns:
        Created page object
    """
    url = f'{NOTION_BASE_URL}/pages'
    payload = {
        'parent': {'database_id': database_id},
        'properties': properties
    }
    if children:
        payload['children'] = children
    
    response = requests.post(url, headers=notion_headers, json=payload)
    
    if response.status_code != 200:
        logger.error(f"Create page failed: {response.status_code} - {response.text}")
        raise Exception(f"Notion page creation failed: {response.status_code}")
    
    return response.json()


# --- Notion page update ---

def update_page(page_id: str, properties: Dict) -> Dict:
    """
    Update an existing Notion page properties.
    
    Args:
        page_id: Page ID to update
        properties: Properties dict to update
    
    Returns:
        Updated page object
    """
    url = f'{NOTION_BASE_URL}/pages/{page_id}'
    payload = {'properties': properties}
    
    response = requests.patch(url, headers=notion_headers, json=payload)
    
    if response.status_code != 200:
        logger.error(f"Update page failed: {response.status_code} - {response.text}")
        raise Exception(f"Notion page update failed: {response.status_code}")
    
    return response.json()


# --- Notion block append ---

def append_blocks(page_id: str, blocks: List[Dict]) -> Dict:
    """
    Append blocks to a Notion page.
    
    Args:
        page_id: Page ID
        blocks: List of block objects
    
    Returns:
        API response
    """
    url = f'{NOTION_BASE_URL}/blocks/{page_id}/children'
    payload = {'children': blocks}
    
    response = requests.patch(url, headers=notion_headers, json=payload)
    
    if response.status_code != 200:
        logger.error(f"Append blocks failed: {response.status_code} - {response.text}")
        raise Exception(f"Notion block append failed: {response.status_code}")
    
    return response.json()


# --- Helper: markdown to Notion blocks ---

def markdown_to_blocks(markdown_text: str, max_chunk_size: int = 2000) -> List[Dict]:
    """
    Convert markdown text to Notion block objects (paragraph blocks).
    Splits large text into chunks to respect Notion's 2000-char limit.
    
    Args:
        markdown_text: Markdown string
        max_chunk_size: Maximum chars per block
    
    Returns:
        List of paragraph block objects
    """
    lines = markdown_text.split('\n')
    blocks = []
    current_chunk = []
    current_length = 0
    
    for line in lines:
        line_length = len(line) + 1  # +1 for newline
        if current_length + line_length > max_chunk_size and current_chunk:
            # Flush current chunk
            text = '\n'.join(current_chunk)
            blocks.append({
                'object': 'block',
                'type': 'paragraph',
                'paragraph': {
                    'rich_text': [{'type': 'text', 'text': {'content': text}}]
                }
            })
            current_chunk = []
            current_length = 0
        
        current_chunk.append(line)
        current_length += line_length
    
    # Flush remaining
    if current_chunk:
        text = '\n'.join(current_chunk)
        blocks.append({
            'object': 'block',
            'type': 'paragraph',
            'paragraph': {
                'rich_text': [{'type': 'text', 'text': {'content': text}}]
            }
        })
    
    return blocks


logger.info("Cell 02 complete: Notion API wrappers and schema helpers ready")


2026-02-06 13:42:57,965 [INFO] Cell 02 complete: Notion API wrappers and schema helpers ready


In [4]:
# ============================================================
# Cell 03 — Time window computation (last 7 days, JST)
# ============================================================
# Overview:
#
# Inputs / Outputs:
#
# Notes:
#

# Compute the time window for weekly digest (last 7 days)
now_jst = datetime.now(TIMEZONE)
week_end = now_jst
week_start = week_end - timedelta(days=7)

# Format for logging and digest title
week_start_str = week_start.strftime('%Y-%m-%d')
week_end_str = week_end.strftime('%Y-%m-%d')
week_label = f"{week_start.strftime('%Y-W%V')}"  # ISO week format

logger.info(f"Time window: {week_start_str} to {week_end_str} ({TIMEZONE})")
logger.info(f"Week label: {week_label}")

# ISO 8601 strings for Notion date filtering
week_start_iso = week_start.isoformat()
week_end_iso = week_end.isoformat()

logger.info("Cell 03 complete: Time window computed")


2026-02-06 13:43:00,092 [INFO] Time window: 2026-01-30 to 2026-02-06 (Asia/Tokyo)
2026-02-06 13:43:00,094 [INFO] Week label: 2026-W05
2026-02-06 13:43:00,095 [INFO] Cell 03 complete: Time window computed


In [5]:
# ============================================================
# Cell 04 — Fetch weekly events from Events data source
# ============================================================
# Overview:
#
# Inputs / Outputs:
#
# Notes:
#

# Build Notion filter for events in the time window
# Filter by Event Date property within [week_start, week_end]
event_filter = {
    'and': [
        {
            'property': 'Date',
            'date': {
                'on_or_after': week_start_iso
            }
        },
        {
            'property': 'Date',
            'date': {
                'on_or_before': week_end_iso
            }
        }
    ]
}

# Optional: Sort by Event Date descending
event_sorts = [
    {
        'property': 'Date',
        'direction': 'descending'
    }
]

logger.info(f"Querying Events database: {NOTION_EVENTS_DB_ID}")
logger.info(f"Filter: Event Date between {week_start_iso} and {week_end_iso}")

# Query the Events database
raw_events = query_database(
    database_id=NOTION_EVENTS_DB_ID,
    filter_obj=event_filter,
    sorts=event_sorts,
    page_size=100
)

logger.info(f"Fetched {len(raw_events)} events from Notion")

# Store raw events for next cell
fetched_events = raw_events

logger.info("Cell 04 complete: Weekly events fetched from Notion")


2026-02-06 13:43:02,283 [INFO] Querying Events database: 2f08e0e4d16280beb40cf607bbb3b828
2026-02-06 13:43:02,285 [INFO] Filter: Event Date between 2026-01-30T13:43:00.091672+09:00 and 2026-02-06T13:43:00.091672+09:00
2026-02-06 13:43:03,683 [INFO] Fetched 72 events from Notion
2026-02-06 13:43:03,684 [INFO] Cell 04 complete: Weekly events fetched from Notion


In [16]:
# ============================================================
# Cell 05 — Normalize records and validate fields (SCHEMA-ALIGNED REPLACEMENT)
# ============================================================
# Overview:
#   Normalize each fetched Notion Event page into a stable python dict using the *actual*
#   Events DB property names defined in EVENTS_SCHEMA:
#     - Name (title)
#     - Date (date)
#     - Detected At (date)
#     - Target (relation)
#     - Event Type (select)
#     - Source URL (url)
#     - Source (select)
#     - Summary (rich_text)
#     - Confidence (number)
#     - Dedup Key (rich_text)
#     - Status (select)
#     - Run ID (rich_text)
#     - Ingested At (date)
#     - Action Needed (checkbox)
#     - Related Papers (relation)
#
# Inputs / Outputs:
#   Inputs:  fetched_events (List[dict]) from Cell 04 (raw Notion pages)
#   Outputs: normalized_events (List[dict])  -> recommended to feed into Cell 06
#
# Notes:
#   - Fixes the root cause of "Untitled/Unknown/blank": wrong property keys/types in prior Cell 05.
#   - Defensive parsing for Notion property types.
#   - Does NOT assume a Domain property exists.
#

from typing import Any, Dict, List, Optional
from collections import defaultdict
import hashlib
import re

def _p(props: Dict[str, Any], name: str) -> Dict[str, Any]:
    """Get property object by name, or empty dict."""
    v = props.get(name, None)
    return v if isinstance(v, dict) else {}

def prop_title(props: Dict[str, Any], name: str) -> str:
    p = _p(props, name)
    arr = p.get("title", [])
    if not isinstance(arr, list):
        return ""
    return "".join([str(x.get("plain_text", "")) for x in arr if isinstance(x, dict)]).strip()

def prop_rich_text(props: Dict[str, Any], name: str) -> str:
    p = _p(props, name)
    arr = p.get("rich_text", [])
    if not isinstance(arr, list):
        return ""
    return "".join([str(x.get("plain_text", "")) for x in arr if isinstance(x, dict)]).strip()

def prop_select(props: Dict[str, Any], name: str) -> str:
    p = _p(props, name)
    sel = p.get("select", None)
    if isinstance(sel, dict):
        return str(sel.get("name", "")).strip()
    return ""

def prop_multi_select(props: Dict[str, Any], name: str) -> List[str]:
    p = _p(props, name)
    arr = p.get("multi_select", [])
    if not isinstance(arr, list):
        return []
    out = []
    for it in arr:
        if isinstance(it, dict):
            nm = str(it.get("name", "")).strip()
            if nm:
                out.append(nm)
    return out

def prop_url(props: Dict[str, Any], name: str) -> str:
    p = _p(props, name)
    v = p.get("url", None)
    return str(v).strip() if v else ""

def prop_number(props: Dict[str, Any], name: str) -> float:
    p = _p(props, name)
    v = p.get("number", None)
    try:
        return float(v) if v is not None else 0.0
    except Exception:
        return 0.0

def prop_checkbox(props: Dict[str, Any], name: str) -> bool:
    p = _p(props, name)
    v = p.get("checkbox", None)
    return bool(v) if v is not None else False

def prop_date(props: Dict[str, Any], name: str) -> str:
    """
    Returns ISO date or datetime string from Notion date property.
    Prefer .start.
    """
    p = _p(props, name)
    d = p.get("date", None)
    if isinstance(d, dict):
        start = d.get("start", None)
        return str(start).strip() if start else ""
    return ""

def prop_relation_ids(props: Dict[str, Any], name: str) -> List[str]:
    p = _p(props, name)
    arr = p.get("relation", [])
    if not isinstance(arr, list):
        return []
    out = []
    for it in arr:
        if isinstance(it, dict):
            rid = str(it.get("id", "")).strip()
            if rid:
                out.append(rid)
    return out

def normalize_date_to_yyyy_mm_dd(dt: str) -> str:
    """
    Normalize:
      - 'YYYY-MM-DD' stays
      - 'YYYY-MM-DDTHH:MM:SS...' -> 'YYYY-MM-DD'
    """
    s = (dt or "").strip()
    if not s:
        return ""
    m = re.match(r"^(\d{4}-\d{2}-\d{2})", s)
    return m.group(1) if m else s

normalized_events: List[Dict[str, Any]] = []
skipped = 0

for page in fetched_events:
    page_id = page.get("id", "")
    props = page.get("properties", {}) or {}

    # --- Extract using your actual schema property names ---
    title = prop_title(props, "Name")
    event_date_raw = prop_date(props, "Date")
    detected_at_raw = prop_date(props, "Detected At")
    target_ids = prop_relation_ids(props, "Target")
    event_type = prop_select(props, "Event Type")
    source_url = prop_url(props, "Source URL")
    source = prop_select(props, "Source")
    summary_text = prop_rich_text(props, "Summary")
    confidence = prop_number(props, "Confidence")
    dedup_key = prop_rich_text(props, "Dedup Key")
    status = prop_select(props, "Status")
    run_id = prop_rich_text(props, "Run ID")
    ingested_at_raw = prop_date(props, "Ingested At")
    action_needed = prop_checkbox(props, "Action Needed")
    related_paper_ids = prop_relation_ids(props, "Related Papers")

    # --- Normalize/clean ---
    title = title if title else "Untitled Event"
    event_date = normalize_date_to_yyyy_mm_dd(event_date_raw)
    detected_at = normalize_date_to_yyyy_mm_dd(detected_at_raw)
    ingested_at = normalize_date_to_yyyy_mm_dd(ingested_at_raw)

    # Fallback: if Detected At missing, use Date (but keep both fields)
    if not detected_at:
        detected_at = event_date

    # Minimal validity: Date OR Detected At must exist
    if not detected_at and not event_date:
        skipped += 1
        logger.warning(f"Skipping page {page_id}: missing both 'Date' and 'Detected At'")
        continue

    # Dedup key fallback: title + detected_at + source_url
    if not dedup_key:
        h_in = f"{title}|{detected_at or event_date}|{source_url or ''}".encode("utf-8")
        dedup_key = hashlib.sha256(h_in).hexdigest()[:16]

    # Compute keywords for clustering (lightweight)
    text_for_kw = f"{title} {summary_text}".strip()
    keywords = extract_keywords(text_for_kw, max_k=12) if "extract_keywords" in globals() else []

    normalized_events.append({
        "notion_page_id": page_id,
        "title": title,
        "detected_at": detected_at,          # YYYY-MM-DD (preferred for weekly filtering)
        "event_date": event_date,            # YYYY-MM-DD
        "target_ids": target_ids,            # relation ids
        "event_type": event_type,            # select name (e.g., VC / STARTUP / POLICY / PEOPLE)
        "source": source,                    # select name
        "source_url": source_url,            # url
        "summary_text": summary_text,        # plain text
        "confidence": confidence,            # float
        "dedup_key": dedup_key,              # rich_text or fallback hash
        "status": status,                    # select name
        "run_id": run_id,                    # rich_text
        "ingested_at": ingested_at,          # YYYY-MM-DD
        "action_needed": action_needed,      # bool
        "related_paper_ids": related_paper_ids,  # relation ids
        "keywords": keywords,                # derived
    })

logger.info(f"Normalized {len(normalized_events)} events (skipped {skipped})")

# Basic stats to sanity-check parsing
if normalized_events:
    blanks_title = sum(1 for e in normalized_events if e["title"] == "Untitled Event")
    blanks_source = sum(1 for e in normalized_events if not str(e.get("source","")).strip())
    blanks_url = sum(1 for e in normalized_events if not str(e.get("source_url","")).strip())
    blanks_sum = sum(1 for e in normalized_events if not str(e.get("summary_text","")).strip())
    empty_targets = sum(1 for e in normalized_events if not e.get("target_ids"))

    logger.info(f"Sanity check: Untitled={blanks_title}, source_blank={blanks_source}, url_blank={blanks_url}, summary_blank={blanks_sum}, target_empty={empty_targets}")

    types = defaultdict(int)
    sources = defaultdict(int)
    for e in normalized_events:
        types[e.get("event_type","")] += 1
        sources[e.get("source","")] += 1
    logger.info(f"Event Type distribution: {dict(types)}")
    logger.info(f"Source distribution (top 10): {dict(list(sorted(sources.items(), key=lambda kv: kv[1], reverse=True))[:10])}")

logger.info("Cell 05 complete: Events normalized and validated (schema-aligned)")


2026-02-06 13:55:05,791 [INFO] Normalized 72 events (skipped 0)
2026-02-06 13:55:05,792 [INFO] Sanity check: Untitled=0, source_blank=0, url_blank=0, summary_blank=0, target_empty=0
2026-02-06 13:55:05,794 [INFO] Event Type distribution: {'POLICY': 15, 'PEOPLE': 54, 'VC': 3}
2026-02-06 13:55:05,795 [INFO] Source distribution (top 10): {'WEB': 72}
2026-02-06 13:55:05,795 [INFO] Cell 05 complete: Events normalized and validated (schema-aligned)


In [18]:
# ============================================================
# Cell 06 — Deduplication and noise filtering (DOMAIN-FREE REPLACEMENT)
# ============================================================
# Overview:
#   Deduplicate by dedup_key and apply noise filters (confidence/status/source).
#   Since Events DB has no "domain" property, we treat Event Type as the domain-like category
#   (VC / STARTUP / POLICY / PEOPLE) for distribution logging.
#
# Inputs / Outputs:
#   Inputs:  normalized_events (List[dict]) from Cell 05
#            MIN_CONFIDENCE_THRESHOLD (float)
#            EXCLUDED_STATUSES (set/list of strings) optional
#            EXCLUDED_SOURCES (set/list of strings) optional (lowercased)
#   Outputs: filtered_events (List[dict])
#
# Notes:
#   - Safe for missing fields
#   - Logs distribution by event_type
#

from typing import Set
from collections import defaultdict

# Safe defaults if not defined earlier
MIN_CONFIDENCE_THRESHOLD = float(globals().get("MIN_CONFIDENCE_THRESHOLD", 0.0))
EXCLUDED_STATUSES = set([s.strip() for s in globals().get("EXCLUDED_STATUSES", set())]) if globals().get("EXCLUDED_STATUSES", None) is not None else set()
EXCLUDED_SOURCES = set([s.strip().lower() for s in globals().get("EXCLUDED_SOURCES", set())]) if globals().get("EXCLUDED_SOURCES", None) is not None else set()

seen_dedup_keys: Set[str] = set()
filtered_events = []

removed_dup = 0
removed_low_conf = 0
removed_status = 0
removed_source = 0

for event in normalized_events:
    dedup_key = str(event.get("dedup_key", "")).strip()
    confidence = float(event.get("confidence") or 0.0)
    status = str(event.get("status", "")).strip()
    source = str(event.get("source", "")).strip()

    # Skip duplicates (first occurrence wins)
    if dedup_key and dedup_key in seen_dedup_keys:
        removed_dup += 1
        continue

    # Skip events below confidence threshold
    if confidence < MIN_CONFIDENCE_THRESHOLD:
        removed_low_conf += 1
        continue

    # Skip excluded statuses (exact match)
    if status and (status in EXCLUDED_STATUSES):
        removed_status += 1
        continue

    # Skip excluded sources (lower match)
    if source and (source.lower() in EXCLUDED_SOURCES):
        removed_source += 1
        continue

    if dedup_key:
        seen_dedup_keys.add(dedup_key)
    filtered_events.append(event)

removed_total = len(normalized_events) - len(filtered_events)
logger.info(
    f"After deduplication and filtering: {len(filtered_events)} events "
    f"(removed {removed_total}; dup={removed_dup}, low_conf={removed_low_conf}, status={removed_status}, source={removed_source})"
)

# Statistics on filtered events
if filtered_events:
    avg_confidence_filtered = sum(float(e.get("confidence") or 0.0) for e in filtered_events) / len(filtered_events)
    logger.info(f"Filtered events average confidence: {avg_confidence_filtered:.2f}")

    # Distribution by Event Type (domain-like)
    type_counts = defaultdict(int)
    for e in filtered_events:
        et = str(e.get("event_type", "")).strip() or "UNKNOWN"
        type_counts[et] += 1
    logger.info(f"Event Type distribution: {dict(type_counts)}")

    # Optional: distribution by Source
    source_counts = defaultdict(int)
    for e in filtered_events:
        src = str(e.get("source", "")).strip() or "UNKNOWN"
        source_counts[src] += 1
    top_sources = dict(list(sorted(source_counts.items(), key=lambda kv: kv[1], reverse=True))[:12])
    logger.info(f"Top Sources (<=12): {top_sources}")
else:
    logger.warning("No events remaining after filtering")

logger.info("Cell 06 complete: Deduplication and noise filtering applied")


2026-02-06 13:56:10,000 [INFO] After deduplication and filtering: 72 events (removed 0; dup=0, low_conf=0, status=0, source=0)
2026-02-06 13:56:10,001 [INFO] Filtered events average confidence: 0.60
2026-02-06 13:56:10,003 [INFO] Event Type distribution: {'POLICY': 15, 'PEOPLE': 54, 'VC': 3}
2026-02-06 13:56:10,004 [INFO] Top Sources (<=12): {'WEB': 72}
2026-02-06 13:56:10,005 [INFO] Cell 06 complete: Deduplication and noise filtering applied


In [19]:
# ============================================================
# Cell 07 — Theme clustering + LLM interpretation (title/summary/why/key events)
# ============================================================
# Overview:
#   Cluster weekly events into coherent themes using a centroid-updating clustering algorithm
#   (targets-first, keywords-second), then (optionally) call OpenAI to generate:
#     - theme_title
#     - theme_summary (2–4 sentences: what happened + what changed)
#     - why_it_matters (<=3 bullets)
#     - key_event_page_ids (3–7 picks)
#   Falls back to deterministic heuristics if LLM is disabled/unavailable.
#
# Inputs / Outputs:
#   Inputs:  filtered_events (List[dict]) from Cell 06
#            OPENAI_API_KEY, llm_model, llm_temperature
#            logger, THEME_OVERLAP_THRESHOLD (float) optional
#   Outputs: themes (List[dict]) with enriched interpretation fields
#
# Notes:
#   - Uses cluster-level centroid features (targets/keywords/types/sources) updated as cluster grows
#   - Robust to missing fields; safe JSON parsing for LLM output; graceful fallback
#   - Does NOT assume a "domain" property exists
#

from __future__ import annotations
from typing import List, Dict, Any, Set, Tuple, Optional
import re
import json
import time
import hashlib

# ----------------------------
# Config knobs (safe defaults)
# ----------------------------
THEME_OVERLAP_THRESHOLD = float(globals().get("THEME_OVERLAP_THRESHOLD", 0.42))  # practical default
MAX_EVENTS_PER_CLUSTER_FOR_LLM = int(globals().get("MAX_EVENTS_PER_CLUSTER_FOR_LLM", 20))
MAX_CLUSTERS_FOR_LLM = int(globals().get("MAX_CLUSTERS_FOR_LLM", 40))            # avoid runaway cost
USE_LLM_LABELING = bool(globals().get("USE_LLM_LABELING", True))
OPENAI_TIMEOUT_SEC = int(globals().get("OPENAI_TIMEOUT_SEC", 45))

# If requests not imported earlier, import here
import requests

# ----------------------------
# Helpers: tokenization/keywords
# ----------------------------
_STOPWORDS = {
    "the","a","an","and","or","to","of","in","on","for","with","as","by","at","from","into","about",
    "is","are","was","were","be","been","being","it","this","that","these","those",
    "will","would","can","could","may","might","should","must",
    "we","they","you","i","he","she","them","us","our","their",
    "new","latest","update","report","reports","announces","announcement","launch","launched",
    "says","said","according","amid","after","before","over","under",
}
def _safe_str(x: Any) -> str:
    return "" if x is None else str(x)

def extract_keywords(text: str, max_k: int = 12) -> List[str]:
    """
    Lightweight keyword extraction:
    - Lowercase
    - Keep alphanumerics and selected separators
    - Remove stopwords
    - Prefer longer tokens
    """
    t = _safe_str(text).lower()
    # split on non-alphanum
    tokens = re.split(r"[^a-z0-9]+", t)
    toks = []
    for tok in tokens:
        if len(tok) < 3:
            continue
        if tok in _STOPWORDS:
            continue
        # drop pure numbers
        if tok.isdigit():
            continue
        toks.append(tok)
    # simple frequency with length bias
    freq: Dict[str, int] = {}
    for tok in toks:
        freq[tok] = freq.get(tok, 0) + 1
    scored = sorted(freq.items(), key=lambda kv: (kv[1], len(kv[0])), reverse=True)
    return [k for k, _ in scored[:max_k]]

def jaccard(a: Set[str], b: Set[str]) -> float:
    if not a and not b:
        return 0.0
    inter = len(a & b)
    union = len(a | b)
    return inter / union if union else 0.0

def stable_hash(s: str) -> str:
    return hashlib.sha256(s.encode("utf-8")).hexdigest()[:16]

# ----------------------------
# Event normalization guardrails
# ----------------------------
def event_card(e: Dict[str, Any]) -> str:
    """
    Compact, LLM-friendly single-line event card.
    Expected keys: notion_page_id, detected_at, event_type, source, title, summary_text, source_url
    """
    page_id = _safe_str(e.get("notion_page_id", e.get("page_id", ""))).strip()
    dt = _safe_str(e.get("detected_at", e.get("event_date", ""))).strip()
    et = _safe_str(e.get("event_type", "")).strip()
    src = _safe_str(e.get("source", "")).strip()
    title = _safe_str(e.get("title", e.get("name", ""))).strip()
    summ = _safe_str(e.get("summary_text", e.get("summary", ""))).strip()
    if len(summ) > 220:
        summ = summ[:220].rstrip() + "…"
    return f"- {page_id} | {dt} | {et} | {src} | {title} :: {summ}"

def get_event_targets(e: Dict[str, Any]) -> Set[str]:
    # supports either target_ids or target_overlap (older code)
    t = e.get("target_ids", None)
    if isinstance(t, list):
        return set([_safe_str(x) for x in t if _safe_str(x)])
    t2 = e.get("target_overlap", None)
    if isinstance(t2, list):
        return set([_safe_str(x) for x in t2 if _safe_str(x)])
    return set()

def get_event_keywords(e: Dict[str, Any]) -> Set[str]:
    # if keywords already present, reuse; else extract from title + summary
    k = e.get("keywords", None)
    if isinstance(k, list) and k:
        return set([_safe_str(x) for x in k if _safe_str(x)])
    text = f"{_safe_str(e.get('title',''))} {_safe_str(e.get('summary_text',''))}"
    return set(extract_keywords(text, max_k=12))

def get_event_type(e: Dict[str, Any]) -> str:
    return _safe_str(e.get("event_type", "")).strip()

def get_event_source(e: Dict[str, Any]) -> str:
    return _safe_str(e.get("source", "")).strip()

def get_event_url(e: Dict[str, Any]) -> str:
    return _safe_str(e.get("source_url", "")).strip()

# ----------------------------
# Cluster representation
# ----------------------------
def make_cluster(seed_event: Dict[str, Any]) -> Dict[str, Any]:
    targets = get_event_targets(seed_event)
    keywords = get_event_keywords(seed_event)
    etype = get_event_type(seed_event)
    source = get_event_source(seed_event)
    return {
        "events": [seed_event],
        "targets": set(targets),
        "keywords": set(keywords),
        "types": set([etype] if etype else []),
        "sources": set([source] if source else []),
        "urls": set([get_event_url(seed_event)] if get_event_url(seed_event) else []),
        "dedup_keys": set([_safe_str(seed_event.get("dedup_key", "")).strip()] if _safe_str(seed_event.get("dedup_key", "")).strip() else []),
    }

def update_cluster(cluster: Dict[str, Any], event: Dict[str, Any]) -> None:
    cluster["events"].append(event)
    cluster["targets"].update(get_event_targets(event))
    cluster["keywords"].update(get_event_keywords(event))
    et = get_event_type(event)
    if et:
        cluster["types"].add(et)
    src = get_event_source(event)
    if src:
        cluster["sources"].add(src)
    url = get_event_url(event)
    if url:
        cluster["urls"].add(url)
    dk = _safe_str(event.get("dedup_key", "")).strip()
    if dk:
        cluster["dedup_keys"].add(dk)

# ----------------------------
# Similarity: event vs cluster
# ----------------------------
def similarity_event_to_cluster(event: Dict[str, Any], cluster: Dict[str, Any]) -> Tuple[float, Dict[str, float]]:
    """
    Targets-first similarity:
      - target overlap strongest (if targets exist)
      - keyword overlap secondary
      - type/source small boosts
      - url/dedup key exact match as strong boost
    """
    e_targets = get_event_targets(event)
    e_keywords = get_event_keywords(event)
    e_type = get_event_type(event)
    e_source = get_event_source(event)
    e_url = get_event_url(event)
    e_dk = _safe_str(event.get("dedup_key", "")).strip()

    # exact identity boosts
    exact_boost = 0.0
    if e_dk and e_dk in cluster["dedup_keys"]:
        exact_boost += 0.35
    if e_url and e_url in cluster["urls"]:
        exact_boost += 0.20

    target_sim = jaccard(e_targets, cluster["targets"]) if (e_targets or cluster["targets"]) else 0.0
    keyword_sim = jaccard(e_keywords, cluster["keywords"]) if (e_keywords or cluster["keywords"]) else 0.0
    type_sim = 1.0 if (e_type and e_type in cluster["types"]) else 0.0
    source_sim = 1.0 if (e_source and e_source in cluster["sources"]) else 0.0

    # If targets exist, weight them heavily. If not, rely more on keywords.
    if e_targets or cluster["targets"]:
        w_target, w_kw, w_type, w_src = 0.62, 0.22, 0.10, 0.06
    else:
        w_target, w_kw, w_type, w_src = 0.10, 0.68, 0.14, 0.08

    score = (
        w_target * target_sim +
        w_kw * keyword_sim +
        w_type * type_sim +
        w_src * source_sim +
        exact_boost
    )
    # cap to 1.0
    score = min(1.0, score)

    parts = {
        "target": target_sim,
        "keyword": keyword_sim,
        "type": type_sim,
        "source": source_sim,
        "exact_boost": exact_boost,
        "score": score
    }
    return score, parts

# ----------------------------
# Clustering: greedy with centroid updates + best-match
# ----------------------------
def cluster_events(events: List[Dict[str, Any]], threshold: float) -> List[Dict[str, Any]]:
    clusters: List[Dict[str, Any]] = []

    # Small ordering trick: place target-rich + higher confidence first (stabilizes clusters)
    def sort_key(e: Dict[str, Any]) -> Tuple[int, float]:
        tcount = len(get_event_targets(e))
        conf = float(e.get("confidence") or 0.0)
        return (-tcount, -conf)

    events_sorted = sorted(events, key=sort_key)

    for e in events_sorted:
        best_idx = None
        best_score = -1.0
        best_parts = None

        for idx, c in enumerate(clusters):
            s, parts = similarity_event_to_cluster(e, c)
            if s > best_score:
                best_score = s
                best_idx = idx
                best_parts = parts

        if best_idx is not None and best_score >= threshold:
            update_cluster(clusters[best_idx], e)
            # optional debug
            # logger.debug(f"Assigned event to cluster {best_idx+1} score={best_score:.3f} parts={best_parts}")
        else:
            clusters.append(make_cluster(e))

    return clusters

logger.info(f"Clustering {len(filtered_events)} filtered events with threshold={THEME_OVERLAP_THRESHOLD:.2f}")
cluster_objs = cluster_events(filtered_events, THEME_OVERLAP_THRESHOLD)
logger.info(f"Created {len(cluster_objs)} clusters")

# ----------------------------
# Deterministic fallback: title + summary for theme
# ----------------------------
def heuristic_theme_title(cluster: Dict[str, Any]) -> str:
    # Prefer targets; else keywords; else types
    if cluster["targets"]:
        # stable: pick the most frequent target in cluster
        freq: Dict[str, int] = {}
        for e in cluster["events"]:
            for t in get_event_targets(e):
                freq[t] = freq.get(t, 0) + 1
        top = sorted(freq.items(), key=lambda kv: kv[1], reverse=True)[0][0]
        t0 = sorted([x for x in cluster["types"] if x])[:1]
        return f"{top} — {t0[0]}" if t0 else f"{top} — Weekly updates"
    if cluster["keywords"]:
        kws = sorted(list(cluster["keywords"]))[:3]
        return " / ".join(kws)
    if cluster["types"]:
        return f"{sorted(list(cluster['types']))[0]} — Weekly cluster"
    return "Weekly theme"

def heuristic_theme_summary(cluster: Dict[str, Any]) -> str:
    # 2–4 sentences, lightweight
    # Use up to 3 representative events
    evs = cluster["events"][:3]
    bullets = []
    for e in evs:
        title = _safe_str(e.get("title","")).strip()
        src = _safe_str(e.get("source","")).strip()
        if title:
            bullets.append(f"- {title}" + (f" ({src})" if src else ""))
    if not bullets:
        return "A cluster of related events was detected this week."
    return "This theme groups related events detected this week:\n" + "\n".join(bullets)

# ----------------------------
# OpenAI call (strict JSON output)
# ----------------------------
def openai_theme_interpretation(cluster: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    if not OPENAI_API_KEY:
        return None

    ev_cards = [event_card(e) for e in cluster["events"][:MAX_EVENTS_PER_CLUSTER_FOR_LLM]]
    content = "\n".join(ev_cards)

    prompt = f"""You are summarizing a weekly digest of cross-domain events (VC/Startup/Policy/People).
Given the event cards, produce a compact theme interpretation.

Return STRICT JSON with keys:
{{
  "theme_title": string,
  "theme_summary": string,
  "why_it_matters": [string, ...],
  "key_event_page_ids": [string, ...],
  "signals": {{
     "novelty": "low|medium|high",
     "momentum": "low|medium|high",
     "uncertainty": "low|medium|high"
  }}
}}

Constraints:
- theme_title: max 8 words
- theme_summary: 2–4 sentences, include "what happened" and "what changed"
- why_it_matters: up to 3 bullets, concrete and decision-relevant
- key_event_page_ids: pick 3–7 page ids from the provided cards; only ids that appear in cards
- Output MUST be valid JSON, no markdown, no extra text

Event cards:
{content}
"""

    headers = {
        "Authorization": f"Bearer {OPENAI_API_KEY}",
        "Content-Type": "application/json"
    }
    payload = {
        "model": llm_model,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": llm_temperature,
        "max_tokens": 450
    }

    resp = requests.post(
        "https://api.openai.com/v1/chat/completions",
        headers=headers,
        json=payload,
        timeout=OPENAI_TIMEOUT_SEC
    )
    if resp.status_code != 200:
        logger.warning(f"OpenAI API error: status={resp.status_code} body={resp.text[:400]}")
        return None

    data = resp.json()
    raw = data["choices"][0]["message"]["content"].strip()

    # Strict JSON parse with a small repair attempt
    try:
        return json.loads(raw)
    except Exception:
        # try to extract JSON object if model wrapped text
        m = re.search(r"\{.*\}", raw, flags=re.DOTALL)
        if not m:
            logger.warning("OpenAI output was not JSON (no object found).")
            return None
        try:
            return json.loads(m.group(0))
        except Exception as e:
            logger.warning(f"Failed to parse JSON from OpenAI output: {e}")
            return None

def validate_llm_theme(obj: Dict[str, Any], cluster: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    """
    Ensure LLM output respects schema + references only known page ids.
    """
    if not isinstance(obj, dict):
        return None

    title = _safe_str(obj.get("theme_title", "")).strip()
    summ = _safe_str(obj.get("theme_summary", "")).strip()
    why = obj.get("why_it_matters", [])
    keys = obj.get("key_event_page_ids", [])
    signals = obj.get("signals", {})

    if not title or not summ:
        return None
    if not isinstance(why, list):
        why = []
    if not isinstance(keys, list):
        keys = []

    known_ids = set(_safe_str(e.get("notion_page_id", e.get("page_id",""))).strip() for e in cluster["events"])
    keys_clean = []
    for k in keys:
        kk = _safe_str(k).strip()
        if kk and kk in known_ids:
            keys_clean.append(kk)
    # If none selected, fallback to top few in cluster
    if len(keys_clean) < 3:
        fallback = []
        for e in cluster["events"]:
            pid = _safe_str(e.get("notion_page_id", e.get("page_id",""))).strip()
            if pid:
                fallback.append(pid)
            if len(fallback) >= 3:
                break
        keys_clean = list(dict.fromkeys(keys_clean + fallback))[:7]

    # normalize signals
    def norm_level(x: Any) -> str:
        s = _safe_str(x).lower().strip()
        return s if s in {"low","medium","high"} else "medium"

    out = {
        "theme_title": " ".join(title.split())[:120],
        "theme_summary": summ.strip(),
        "why_it_matters": [ _safe_str(x).strip() for x in why if _safe_str(x).strip() ][:3],
        "key_event_page_ids": keys_clean[:7],
        "signals": {
            "novelty": norm_level(signals.get("novelty", "medium")),
            "momentum": norm_level(signals.get("momentum", "medium")),
            "uncertainty": norm_level(signals.get("uncertainty", "medium")),
        }
    }
    return out

# ----------------------------
# Build final themes list
# ----------------------------
themes: List[Dict[str, Any]] = []
llm_calls = 0
start_t = time.time()

# Sort clusters by "importance proxy" (size * avg confidence * target richness)
def cluster_importance(c: Dict[str, Any]) -> float:
    n = len(c["events"])
    avg_conf = 0.0
    if n:
        avg_conf = sum(float(e.get("confidence") or 0.0) for e in c["events"]) / n
    t_rich = len(c["targets"])
    return (n ** 1.1) * (1.0 + avg_conf) * (1.0 + 0.15 * t_rich)

cluster_objs_sorted = sorted(cluster_objs, key=cluster_importance, reverse=True)

for idx, c in enumerate(cluster_objs_sorted, 1):
    # Stable theme id: hash of top evidence ids + window-agnostic signals
    ids_for_hash = []
    for e in c["events"][:10]:
        pid = _safe_str(e.get("notion_page_id", e.get("page_id",""))).strip()
        if pid:
            ids_for_hash.append(pid)
    base = "|".join(ids_for_hash) if ids_for_hash else f"cluster_{idx}_{len(c['events'])}"
    theme_id = stable_hash(base)

    # Default heuristic interpretation
    theme_title = heuristic_theme_title(c)
    theme_summary = heuristic_theme_summary(c)
    why_it_matters = []
    key_event_page_ids = []
    for e in c["events"][:5]:
        pid = _safe_str(e.get("notion_page_id", e.get("page_id",""))).strip()
        if pid:
            key_event_page_ids.append(pid)
    key_event_page_ids = list(dict.fromkeys(key_event_page_ids))[:7]
    signals = {"novelty": "medium", "momentum": "medium", "uncertainty": "medium"}

    # LLM enrich (only if enabled and cluster has enough info)
    if USE_LLM_LABELING and OPENAI_API_KEY and llm_calls < MAX_CLUSTERS_FOR_LLM and len(c["events"]) >= 2:
        llm_obj = None
        try:
            llm_raw = openai_theme_interpretation(c)
            if llm_raw:
                llm_obj = validate_llm_theme(llm_raw, c)
        except Exception as e:
            logger.warning(f"LLM interpretation failed for cluster {idx}: {e}")
            llm_obj = None

        if llm_obj:
            theme_title = llm_obj["theme_title"]
            theme_summary = llm_obj["theme_summary"]
            why_it_matters = llm_obj["why_it_matters"]
            key_event_page_ids = llm_obj["key_event_page_ids"]
            signals = llm_obj["signals"]
            llm_calls += 1

    # Aggregate cluster meta
    types = sorted([t for t in c["types"] if t])
    sources = sorted([s for s in c["sources"] if s])
    targets = sorted([t for t in c["targets"] if t])

    # Provide compact keyword set (top by frequency across events)
    kw_freq: Dict[str, int] = {}
    for e in c["events"]:
        for k in get_event_keywords(e):
            kw_freq[k] = kw_freq.get(k, 0) + 1
    top_keywords = [k for k, _ in sorted(kw_freq.items(), key=lambda kv: kv[1], reverse=True)[:18]]

    themes.append({
        "theme_id": theme_id,
        "theme_title": theme_title,
        "theme_summary": theme_summary,
        "why_it_matters": why_it_matters,
        "signals": signals,
        "event_count": len(c["events"]),
        "key_event_page_ids": key_event_page_ids,
        "targets": targets,
        "types": types,
        "sources": sources,
        "keywords": top_keywords,
        "events": c["events"],  # full evidence
    })

elapsed = time.time() - start_t
logger.info(f"Built {len(themes)} themes (LLM calls={llm_calls}) in {elapsed:.1f}s")
for t in themes[:10]:
    logger.info(f"  Theme: {t['theme_title']} | events={t['event_count']} | signals={t['signals']}")

logger.info("Cell 07 complete: Theme clustering + interpretation done")


2026-02-06 13:56:14,769 [INFO] Clustering 72 filtered events with threshold=0.40
2026-02-06 13:56:14,776 [INFO] Created 9 clusters
2026-02-06 13:56:51,947 [INFO] Built 9 themes (LLM calls=7) in 37.2s
2026-02-06 13:56:51,948 [INFO]   Theme: Nvidia's Strategic Moves in AI Investment | events=40 | signals={'novelty': 'medium', 'momentum': 'high', 'uncertainty': 'medium'}
2026-02-06 13:56:51,949 [INFO]   Theme: AI Leadership and Market Dynamics | events=14 | signals={'novelty': 'medium', 'momentum': 'high', 'uncertainty': 'medium'}
2026-02-06 13:56:51,949 [INFO]   Theme: Advancements in UK Innovation and Policy | events=7 | signals={'novelty': 'medium', 'momentum': 'high', 'uncertainty': 'medium'}
2026-02-06 13:56:51,951 [INFO]   Theme: Updates in Health Policy and Guidance | events=3 | signals={'novelty': 'medium', 'momentum': 'high', 'uncertainty': 'medium'}
2026-02-06 13:56:51,952 [INFO]   Theme: NIH Policy Changes Amid Funding Lapse | events=2 | signals={'novelty': 'medium', 'momentum'

In [20]:
# ============================================================
# Cell 07a — Quick preview of themes
# ============================================================
# Overview:
#   Quick human-readable preview of what Cell 07 produced.
#
# Inputs / Outputs:
#   Inputs: themes (List[dict])
#   Outputs: print preview
#
# Notes:
#   - Prints top themes + key events + small sample event cards
#

def _event_by_page_id(themes, page_id: str):
    for t in themes:
        for e in t.get("events", []):
            pid = str(e.get("notion_page_id", e.get("page_id",""))).strip()
            if pid == page_id:
                return e
    return None

print("\n" + "="*80)
print(f"THEMES PREVIEW  (themes={len(themes)})")
print("="*80)

for i, t in enumerate(themes[:10], 1):
    print(f"\n[{i}] {t.get('theme_title','(no title)')}")
    print(f"  theme_id     : {t.get('theme_id')}")
    print(f"  event_count  : {t.get('event_count')}")
    print(f"  signals      : {t.get('signals')}")
    if t.get("targets"):
        print(f"  targets (top): {t.get('targets')[:8]}{' ...' if len(t.get('targets'))>8 else ''}")
    if t.get("types"):
        print(f"  types        : {t.get('types')}")
    if t.get("sources"):
        print(f"  sources (top): {t.get('sources')[:8]}{' ...' if len(t.get('sources'))>8 else ''}")
    if t.get("keywords"):
        print(f"  keywords     : {t.get('keywords')[:12]}{' ...' if len(t.get('keywords'))>12 else ''}")

    print("\n  Summary:")
    print("  " + str(t.get("theme_summary","")).replace("\n", "\n  "))

    if t.get("why_it_matters"):
        print("\n  Why it matters:")
        for b in t["why_it_matters"]:
            print(f"  - {b}")

    # Show key events with titles/urls
    print("\n  Key events:")
    key_ids = t.get("key_event_page_ids", [])[:7]
    if not key_ids:
        # fallback: show first few
        key_ids = [str(e.get("notion_page_id", e.get("page_id",""))).strip() for e in t.get("events", [])[:5] if str(e.get("notion_page_id", e.get("page_id",""))).strip()]
    for pid in key_ids:
        e = _event_by_page_id(themes, pid)
        if not e:
            print(f"  - {pid} (not found in theme events list)")
            continue
        title = str(e.get("title","")).strip()
        et = str(e.get("event_type","")).strip()
        src = str(e.get("source","")).strip()
        url = str(e.get("source_url","")).strip()
        dt = str(e.get("detected_at", e.get("event_date",""))).strip()
        print(f"  - {dt} | {et} | {src} | {title}")
        if url:
            print(f"    {url}")

    # Show a small sample of event cards
    print("\n  Sample events (first 5 in cluster):")
    for e in t.get("events", [])[:5]:
        pid = str(e.get("notion_page_id", e.get("page_id",""))).strip()
        dt = str(e.get("detected_at", e.get("event_date",""))).strip()
        et = str(e.get("event_type","")).strip()
        src = str(e.get("source","")).strip()
        title = str(e.get("title","")).strip()
        summ = str(e.get("summary_text", e.get("summary",""))).strip()
        if len(summ) > 160:
            summ = summ[:160].rstrip() + "…"
        print(f"  - {pid} | {dt} | {et} | {src} | {title} :: {summ}")

print("\n" + "="*80)



THEMES PREVIEW  (themes=9)

[1] Nvidia's Strategic Moves in AI Investment
  theme_id     : d3aaabbdf6b2c2f9
  event_count  : 40
  signals      : {'novelty': 'medium', 'momentum': 'high', 'uncertainty': 'medium'}
  targets (top): ['2fb8e0e4-d162-81f9-aecc-f19a28fad58b']
  types        : ['PEOPLE']
  sources (top): ['WEB']
  keywords     : ['nvidia', 'jensen', 'ceo', 'huang', 'openai', 'investment', 'company', 'billion', 'september', 'denies', 'partnership', 'chipmaker'] ...

  Summary:
  Nvidia CEO Jensen Huang has reaffirmed the company's commitment to invest in OpenAI, dismissing rumors of a stalled deal. This comes amid a broader context of declining software stocks, which Huang attributes to misconceptions about AI replacing existing tools. The collaboration with OpenAI is positioned as a significant step for Nvidia as it navigates the evolving AI landscape.

  Why it matters:
  - Nvidia's investment in OpenAI could reshape the AI market and influence future tech developments.
  - 

In [21]:
# ============================================================
# Cell 08 — Theme scoring + Top N selection (SCHEMA-ALIGNED REPLACEMENT)
# ============================================================
# Overview:
#   Score each theme produced in Cell 07 and select Top N themes + Top K events per theme.
#   This replacement matches the new theme structure:
#     - theme_title (not "label")
#     - targets/types/sources/keywords present
#     - signals present (novelty/momentum/uncertainty)
#   Also fixes the old "domain" dependency (not available).
#
# Inputs / Outputs:
#   Inputs:  themes (List[dict]) from Cell 07
#            week_start, week_end (datetime) from Cell 03
#            TIMEZONE (tzinfo) from Cell 03
#            TOP_N_THEMES (int) optional
#            MAX_EVENTS_PER_THEME (int) optional
#   Outputs: themes_sorted (List[dict])
#            top_themes (List[dict]) each with:
#               - score
#               - score_breakdown
#               - top_events (List[dict])
#
# Notes:
#   - Uses event_count, avg_confidence, target diversity, type diversity, recency,
#     and LLM signals as lightweight multipliers.
#   - If parsing dates fails, recency defaults to mid-week.
#

import math
import statistics
from datetime import datetime

TOP_N_THEMES = int(globals().get("TOP_N_THEMES", 8))
MAX_EVENTS_PER_THEME = int(globals().get("MAX_EVENTS_PER_THEME", 8))

def _to_dt_safe(date_str: str) -> datetime | None:
    """
    Parse YYYY-MM-DD or ISO datetime. Returns naive datetime.
    We interpret it in TIMEZONE later.
    """
    s = (date_str or "").strip()
    if not s:
        return None
    # normalize Z
    s = s.replace("Z", "+00:00")
    try:
        # If YYYY-MM-DD, fromisoformat works
        return datetime.fromisoformat(s)
    except Exception:
        # Try take first 10 chars
        try:
            return datetime.fromisoformat(s[:10])
        except Exception:
            return None

def _recency_score(events: list) -> float:
    """
    Returns [0..1] where 1 = closer to week_end, 0 = closer to week_start.
    """
    if not events:
        return 0.5
    scores = []
    for e in events:
        ds = (e.get("detected_at") or e.get("event_date") or "").strip()
        dt = _to_dt_safe(ds)
        if not dt:
            scores.append(0.5)
            continue
        try:
            # treat naive date as TIMEZONE local
            dt_local = dt
            if dt_local.tzinfo is None:
                dt_local = dt_local.replace(tzinfo=TIMEZONE)
            else:
                dt_local = dt_local.astimezone(TIMEZONE)

            span = (week_end - week_start).total_seconds()
            if span <= 0:
                scores.append(0.5)
                continue
            pos = (dt_local - week_start).total_seconds() / span
            scores.append(max(0.0, min(1.0, pos)))
        except Exception:
            scores.append(0.5)
    return statistics.mean(scores) if scores else 0.5

def _signal_multiplier(signals: dict) -> float:
    """
    Convert LLM signals into a mild multiplier.
    momentum high should raise; uncertainty high should slightly lower.
    """
    if not isinstance(signals, dict):
        return 1.0
    m = str(signals.get("momentum", "medium")).lower()
    n = str(signals.get("novelty", "medium")).lower()
    u = str(signals.get("uncertainty", "medium")).lower()

    def level(x: str) -> float:
        if x == "high":
            return 1.10
        if x == "low":
            return 0.95
        return 1.00

    mult = 1.0
    mult *= level(m)
    mult *= level(n)
    # uncertainty reduces slightly when high
    if u == "high":
        mult *= 0.92
    elif u == "low":
        mult *= 1.03
    return mult

def score_theme(theme: Dict[str, Any]) -> tuple[float, Dict[str, float]]:
    events = theme.get("events", [])
    event_count = int(theme.get("event_count") or len(events) or 0)
    if event_count <= 0:
        return 0.0, {"count": 0, "conf": 0, "targets": 0, "types": 0, "recency": 0, "signals": 1.0}

    # 1) Count score (log-scaled)
    count_score = math.log1p(event_count) * 10.0  # ~0..40ish

    # 2) Confidence score (mean confidence)
    confs = []
    for e in events:
        try:
            confs.append(float(e.get("confidence") or 0.0))
        except Exception:
            confs.append(0.0)
    avg_conf = statistics.mean(confs) if confs else 0.0
    confidence_score = avg_conf * 20.0  # 0..20

    # 3) Target diversity (unique targets)
    targets = theme.get("targets", []) or []
    target_div = min(len(targets), 10)  # cap
    target_score = target_div * 2.5     # 0..25

    # 4) Type diversity (VC/STARTUP/POLICY/PEOPLE) as "domain-like"
    types = theme.get("types", []) or []
    type_div = min(len(types), 4)
    type_score = (type_div - 1) * 4.0 if type_div > 1 else 0.0  # 0..12 bonus

    # 5) Recency
    rec = _recency_score(events)        # 0..1
    recency_score = rec * 10.0          # 0..10

    # 6) LLM signals multiplier
    sig_mult = _signal_multiplier(theme.get("signals", {}))

    base = count_score + confidence_score + target_score + type_score + recency_score
    total = base * sig_mult

    breakdown = {
        "count_score": round(count_score, 3),
        "confidence_score": round(confidence_score, 3),
        "target_score": round(target_score, 3),
        "type_score": round(type_score, 3),
        "recency_score": round(recency_score, 3),
        "signals_multiplier": round(sig_mult, 3),
        "base": round(base, 3),
    }
    return total, breakdown

logger.info("Scoring themes...")
for t in themes:
    score, breakdown = score_theme(t)
    t["score"] = float(score)
    t["score_breakdown"] = breakdown
    logger.info(f"  {t.get('theme_title','(no title)')}: score={t['score']:.2f} breakdown={breakdown}")

# Sort by score descending
themes_sorted = sorted(themes, key=lambda x: float(x.get("score", 0.0)), reverse=True)

# Select Top N themes
top_themes = themes_sorted[:TOP_N_THEMES]

logger.info(f"Selected top {len(top_themes)} themes (TOP_N_THEMES={TOP_N_THEMES}):")
for i, t in enumerate(top_themes, 1):
    logger.info(f"  {i}. {t.get('theme_title','(no title)')} | events={t.get('event_count')} | score={t.get('score'):.2f}")

# For each top theme, select top events:
# priority: (confidence desc) then (has_summary) then (has_url)
def _event_rank_key(e: Dict[str, Any]) -> tuple:
    conf = float(e.get("confidence") or 0.0)
    has_sum = 1 if str(e.get("summary_text","")).strip() else 0
    has_url = 1 if str(e.get("source_url","")).strip() else 0
    return (conf, has_sum, has_url)

for t in top_themes:
    evs = t.get("events", []) or []
    evs_sorted = sorted(evs, key=_event_rank_key, reverse=True)
    t["top_events"] = evs_sorted[:MAX_EVENTS_PER_THEME]
    if len(evs) > MAX_EVENTS_PER_THEME:
        logger.info(f"  Theme '{t.get('theme_title','(no title)')}': showing top {MAX_EVENTS_PER_THEME} of {len(evs)} events")

logger.info("Cell 08 complete: Theme scoring and selection done")


2026-02-06 14:00:26,862 [INFO] Scoring themes...
2026-02-06 14:00:26,870 [INFO]   Nvidia's Strategic Moves in AI Investment: score=63.21 breakdown={'count_score': 37.136, 'confidence_score': 12.0, 'target_score': 2.5, 'type_score': 0.0, 'recency_score': 5.826, 'signals_multiplier': 1.1, 'base': 57.462}
2026-02-06 14:00:26,871 [INFO]   AI Leadership and Market Dynamics: score=55.17 breakdown={'count_score': 27.081, 'confidence_score': 12.0, 'target_score': 2.5, 'type_score': 0.0, 'recency_score': 8.571, 'signals_multiplier': 1.1, 'base': 50.152}
2026-02-06 14:00:26,873 [INFO]   Advancements in UK Innovation and Policy: score=45.56 breakdown={'count_score': 20.794, 'confidence_score': 12.0, 'target_score': 2.5, 'type_score': 0.0, 'recency_score': 6.122, 'signals_multiplier': 1.1, 'base': 41.417}
2026-02-06 14:00:26,874 [INFO]   Updates in Health Policy and Guidance: score=40.25 breakdown={'count_score': 13.863, 'confidence_score': 12.0, 'target_score': 2.5, 'type_score': 0.0, 'recency_sc

In [22]:
# ============================================================
# Cell 09 — Render weekly digest markdown (SCHEMA-ALIGNED REPLACEMENT + END SUMMARY)
# ============================================================
# Overview:
#   Render a weekly digest markdown from top_themes (Cell 08).
#   Includes:
#     - Header (period, timezone, generated time, run_id)
#     - Quick stats + Event Type breakdown
#     - Executive summary (LLM-derived theme summaries)
#     - Top themes with: signals, score, why-it-matters, key events, targets, keywords
#     - A compact "At-a-glance" summary at the END (as requested)
#
# Inputs / Outputs:
#   Inputs:  filtered_events (List[dict]) from Cell 06
#            themes (List[dict]) from Cell 07
#            top_themes (List[dict]) from Cell 08
#            week_start, week_end, TIMEZONE, now_jst, RUN_ID from earlier cells
#   Outputs: weekly_digest_md (str)
#
# Notes:
#   - Uses event_type (VC/STARTUP/POLICY/PEOPLE) as "domain-like" category.
#   - Uses schema-aligned fields: source_url, summary_text, detected_at/event_date, theme_title, etc.
#

from typing import Dict, List, Any
from collections import defaultdict

def _md_escape(s: str) -> str:
    # Minimal markdown escaping for brackets
    return (s or "").replace("[", "\\[").replace("]", "\\]")

def _short(s: str, n: int = 180) -> str:
    s = (s or "").strip()
    if len(s) <= n:
        return s
    return s[:n].rstrip() + "…"

def render_event_row(event: Dict[str, Any]) -> str:
    """
    Render a single event as a markdown list item.
    Format:
      - **[Title](URL)** (Type | Source) — YYYY-MM-DD  (conf=0.60)
        Summary...
    """
    title = _md_escape(str(event.get("title","")).strip() or "Untitled")
    url = str(event.get("source_url","")).strip()
    et = str(event.get("event_type","")).strip() or "UNKNOWN"
    src = str(event.get("source","")).strip() or "UNKNOWN"
    dt = str(event.get("detected_at") or event.get("event_date") or "").strip()
    dt = dt[:10] if dt else ""
    conf = float(event.get("confidence") or 0.0)
    summ = _short(str(event.get("summary_text","")).strip(), 200)

    if url:
        title_md = f"**[{title}]({url})**"
    else:
        title_md = f"**{title}**"

    line = f"- {title_md} ({et} | {src})"
    if dt:
        line += f" — {dt}"
    line += f"  (conf={conf:.2f})"
    if summ:
        line += f"\n  {_md_escape(summ)}"
    return line

def render_theme_section(theme: Dict[str, Any], rank: int) -> str:
    """
    Render a single theme section with strong interpretability.
    """
    lines: List[str] = []

    title = theme.get("theme_title", "(no title)")
    lines.append(f"## {rank}. {title}")
    lines.append("")

    # Metadata
    event_count = int(theme.get("event_count") or 0)
    score = float(theme.get("score") or 0.0)
    signals = theme.get("signals", {}) or {}
    novelty = str(signals.get("novelty","")).lower() or "medium"
    momentum = str(signals.get("momentum","")).lower() or "medium"
    uncertainty = str(signals.get("uncertainty","")).lower() or "medium"

    types = theme.get("types", []) or []
    sources = theme.get("sources", []) or []
    targets = theme.get("targets", []) or []
    keywords = theme.get("keywords", []) or []

    types_str = ", ".join(types[:6]) + (f" (+{len(types)-6})" if len(types) > 6 else "")
    sources_str = ", ".join(sources[:6]) + (f" (+{len(sources)-6})" if len(sources) > 6 else "")
    targets_str = ", ".join(targets[:5]) + (f" (+{len(targets)-5})" if len(targets) > 5 else "")
    kw_str = ", ".join(keywords[:12]) + (f" (+{len(keywords)-12})" if len(keywords) > 12 else "")

    lines.append(f"**Events:** {event_count}  |  **Score:** {score:.2f}")
    lines.append(f"**Signals:** novelty={novelty}, momentum={momentum}, uncertainty={uncertainty}")
    if types_str:
        lines.append(f"**Types:** {types_str}")
    if sources_str:
        lines.append(f"**Sources:** {sources_str}")
    if targets_str.strip(", "):
        lines.append(f"**Targets:** {targets_str}")
    if kw_str.strip(", "):
        lines.append(f"**Keywords:** {kw_str}")
    lines.append("")

    # LLM summary + why it matters
    summ = str(theme.get("theme_summary","")).strip()
    if summ:
        lines.append("### Theme Summary")
        lines.append("")
        lines.append(_md_escape(summ))
        lines.append("")

    why = theme.get("why_it_matters", []) or []
    if why:
        lines.append("### Why it matters")
        lines.append("")
        for b in why[:3]:
            b = str(b).strip()
            if b:
                lines.append(f"- {_md_escape(b)}")
        lines.append("")

    # Key events: prefer theme["top_events"] if present, else fallback to key_event_page_ids
    lines.append("### Key Events")
    lines.append("")

    top_events = theme.get("top_events", None)
    if isinstance(top_events, list) and top_events:
        for e in top_events:
            lines.append(render_event_row(e))
            lines.append("")
    else:
        # fallback: show first few raw events
        for e in (theme.get("events", []) or [])[:6]:
            lines.append(render_event_row(e))
            lines.append("")

    lines.append("---")
    lines.append("")
    return "\n".join(lines)

# ----------------------------
# Build digest markdown
# ----------------------------
week_start_str = week_start.strftime("%Y-%m-%d")
week_end_str = week_end.strftime("%Y-%m-%d")
week_label = f"{week_start_str}–{week_end_str}"

digest_lines: List[str] = []

# Header
digest_lines.append(f"# Weekly Events Digest: {week_label}")
digest_lines.append("")
digest_lines.append(f"**Period:** {week_start_str} to {week_end_str}")
digest_lines.append(f"**Timezone:** {TIMEZONE}")
digest_lines.append(f"**Generated:** {now_jst.strftime('%Y-%m-%d %H:%M:%S %Z')}")
digest_lines.append(f"**Run ID:** {RUN_ID}")
digest_lines.append("")
digest_lines.append("---")
digest_lines.append("")

# Stats
total_events = len(filtered_events)
total_themes = len(themes)
top_n = len(top_themes)

digest_lines.append("## Weekly Coverage")
digest_lines.append("")
digest_lines.append(f"- **Events processed (post-filter):** {total_events}")
digest_lines.append(f"- **Themes discovered:** {total_themes}")
digest_lines.append(f"- **Top themes selected:** {top_n}")
digest_lines.append("")

# Event Type breakdown (domain-like)
type_breakdown = defaultdict(int)
for e in filtered_events:
    et = str(e.get("event_type","")).strip() or "UNKNOWN"
    type_breakdown[et] += 1

digest_lines.append("### Event Type Breakdown")
digest_lines.append("")
for et, cnt in sorted(type_breakdown.items(), key=lambda kv: kv[1], reverse=True):
    digest_lines.append(f"- {et}: {cnt}")
digest_lines.append("")
digest_lines.append("---")
digest_lines.append("")

# Executive Summary: stitch top themes' summaries
digest_lines.append("## Executive Summary")
digest_lines.append("")
if top_themes:
    for i, t in enumerate(top_themes, 1):
        title = t.get("theme_title","(no title)")
        summ = _short(str(t.get("theme_summary","")).strip(), 260)
        sig = t.get("signals", {}) or {}
        digest_lines.append(f"- **{i}. {title}**  _(momentum={sig.get('momentum','medium')}, novelty={sig.get('novelty','medium')}, uncertainty={sig.get('uncertainty','medium')})_")
        if summ:
            digest_lines.append(f"  - {_md_escape(summ)}")
else:
    digest_lines.append("No themes available.")
digest_lines.append("")
digest_lines.append("---")
digest_lines.append("")

# Top Themes sections
digest_lines.append(f"## Top {top_n} Themes")
digest_lines.append("")
for rank, theme in enumerate(top_themes, 1):
    digest_lines.append(render_theme_section(theme, rank))

# Footer + END summary (as requested)
digest_lines.append("## At-a-glance (End Summary)")
digest_lines.append("")
if top_themes:
    digest_lines.append("**Key takeaways (one-liners):**")
    for i, t in enumerate(top_themes, 1):
        one = _short(str(t.get("theme_summary","")).strip(), 140)
        digest_lines.append(f"- {i}. **{t.get('theme_title','(no title)')}** — {_md_escape(one) if one else 'See details above.'}")
    digest_lines.append("")
    digest_lines.append("**Recommended actions (draft):**")
    # Derive a few generic actions based on signals/types (lightweight, deterministic)
    high_momentum = [t for t in top_themes if str((t.get("signals",{}) or {}).get("momentum","")).lower() == "high"]
    policy_heavy = [t for t in top_themes if "POLICY" in (t.get("types") or [])]
    people_heavy = [t for t in top_themes if "PEOPLE" in (t.get("types") or [])]
    vc_heavy = [t for t in top_themes if "VC" in (t.get("types") or [])]

    if high_momentum:
        digest_lines.append(f"- Track **{len(high_momentum)} high-momentum** theme(s) next week; keep monitoring sources and targets.")
    if policy_heavy:
        digest_lines.append(f"- For **POLICY** themes ({len(policy_heavy)}), extract concrete regulatory changes + affected stakeholders.")
    if people_heavy:
        digest_lines.append(f"- For **PEOPLE** themes ({len(people_heavy)}), confirm role changes and link to org strategy shifts.")
    if vc_heavy:
        digest_lines.append(f"- For **VC** themes ({len(vc_heavy)}), note fund strategy shifts and potential startup opportunities.")
    if not (high_momentum or policy_heavy or people_heavy or vc_heavy):
        digest_lines.append("- Review top themes and decide what to read / monitor next week.")
else:
    digest_lines.append("No top themes were selected.")
digest_lines.append("")
digest_lines.append("---")
digest_lines.append("")
digest_lines.append(f"*Generated by Weekly Events Digest pipeline (run_id: {RUN_ID})*")
digest_lines.append("")

weekly_digest_md = "\n".join(digest_lines)

logger.info(f"Rendered digest markdown: {len(weekly_digest_md)} characters")
logger.info(f"Top themes included: {len(top_themes)}")
logger.info("Cell 09 complete: Weekly digest markdown rendered")


2026-02-06 14:02:39,761 [INFO] Rendered digest markdown: 21415 characters
2026-02-06 14:02:39,762 [INFO] Top themes included: 5
2026-02-06 14:02:39,764 [INFO] Cell 09 complete: Weekly digest markdown rendered


In [23]:
# ============================================================
# Cell 10 — Notion writeback: upsert Weekly Digest page (DATA_SOURCES API, SAFE CONTENT REPLACE)
# ============================================================
# Overview:
#   Upsert weekly digest page into Notion Weekly Digests data source (NOTION_WEEKLY_DIGESTS_DS_ID).
#   Strategy (idempotent + safe):
#     1) Find an existing page by deterministic title key: "Weekly Events Digest: <week_label>"
#        using the data_sources query endpoint with a title "contains" filter.
#     2) If found: update title + replace content by appending a single "markdown dump" block.
#        (We do NOT delete blocks because Notion doesn't support DELETE block reliably for all blocks.)
#        Instead: append a divider + toggle block with the new digest for auditability.
#     3) If not found: create new page, then append content blocks.
#     4) Skip gracefully if env var not set.
#
# Inputs / Outputs:
#   Inputs:  weekly_digest_md (str) from Cell 09
#            NOTION_TOKEN, NOTION_WEEKLY_DIGESTS_DS_ID (env)
#            now_jst, week_label, week_start_str, week_end_str, RUN_ID
#   Outputs: notion_digest_page_id (str|None)
#            notion_digest_page_url (str|None)
#
# Notes:
#   - Assumes Notion API >= 2025-09-03 and uses /v1/data_sources/{id}/query.
#   - Does NOT assume Weekly Digests DB has any properties beyond a title property.
#   - Stores all digest content in page blocks. Properties set: title only (safe).
#   - If you later add properties like Week/Start Date/End Date, you can extend build_properties().
#

import os
import re
import requests
from typing import Any, Dict, List, Optional

NOTION_WEEKLY_DIGESTS_DS_ID = os.getenv("NOTION_WEEKLY_DIGESTS_DS_ID")

NOTION_BASE_URL = "https://api.notion.com/v1"
NOTION_VERSION = "2025-09-03"

notion_headers = {
    "Authorization": f"Bearer {NOTION_TOKEN}",
    "Notion-Version": NOTION_VERSION,
    "Content-Type": "application/json",
}

def _notion_req(method: str, path: str, json_body: Optional[Dict[str, Any]] = None, timeout: int = 45) -> Dict[str, Any]:
    url = f"{NOTION_BASE_URL}{path}"
    resp = requests.request(method, url, headers=notion_headers, json=json_body, timeout=timeout)
    if resp.status_code < 200 or resp.status_code >= 300:
        raise RuntimeError(f"Notion API error {resp.status_code} on {method} {path}: {resp.text[:800]}")
    return resp.json() if resp.text else {}

def query_data_source(data_source_id: str, filter_obj: Optional[Dict[str, Any]] = None, sorts: Optional[List[Dict[str, Any]]] = None,
                      page_size: int = 50, start_cursor: Optional[str] = None) -> Dict[str, Any]:
    body: Dict[str, Any] = {"page_size": page_size}
    if filter_obj:
        body["filter"] = filter_obj
    if sorts:
        body["sorts"] = sorts
    if start_cursor:
        body["start_cursor"] = start_cursor
    return _notion_req("POST", f"/data_sources/{data_source_id}/query", body)

def create_page(parent_data_source_id: str, properties: Dict[str, Any], children_blocks: Optional[List[Dict[str, Any]]] = None) -> Dict[str, Any]:
    body: Dict[str, Any] = {
        "parent": {"data_source_id": parent_data_source_id},
        "properties": properties,
    }
    if children_blocks:
        body["children"] = children_blocks
    return _notion_req("POST", "/pages", body)

def update_page(page_id: str, properties: Optional[Dict[str, Any]] = None, archive: Optional[bool] = None) -> Dict[str, Any]:
    body: Dict[str, Any] = {}
    if properties is not None:
        body["properties"] = properties
    if archive is not None:
        body["archived"] = archive
    return _notion_req("PATCH", f"/pages/{page_id}", body)

def append_blocks(block_id: str, children_blocks: List[Dict[str, Any]]) -> Dict[str, Any]:
    # append children blocks to page (page is also a block)
    body = {"children": children_blocks}
    return _notion_req("PATCH", f"/blocks/{block_id}/children", body)

def _title_prop(text: str) -> Dict[str, Any]:
    return {"title": [{"type": "text", "text": {"content": text}}]}

def _find_title_property_key(page_obj: Dict[str, Any]) -> str:
    """
    Notion DBs can name the title property differently (e.g., 'Name' or 'Title').
    The property object includes type == 'title'. Find that key.
    """
    props = page_obj.get("properties", {}) or {}
    for k, v in props.items():
        if isinstance(v, dict) and v.get("type") == "title":
            return k
    # fallback guesses
    for guess in ["Name", "Title"]:
        if guess in props:
            return guess
    return "Name"

def _build_title_properties(title_key: str, title_text: str) -> Dict[str, Any]:
    return {title_key: _title_prop(title_text)}

def _chunk_blocks(blocks: List[Dict[str, Any]], n: int = 100) -> List[List[Dict[str, Any]]]:
    return [blocks[i:i+n] for i in range(0, len(blocks), n)]

def _markdown_to_blocks_minimal(md: str, max_chars_per_block: int = 1800) -> List[Dict[str, Any]]:
    """
    Minimal block conversion:
    - Store markdown as a few code blocks (plain_text) to preserve content.
    - Safer than trying to fully parse markdown to Notion blocks.
    """
    md = (md or "").strip()
    if not md:
        return [{
            "object": "block",
            "type": "paragraph",
            "paragraph": {"rich_text": [{"type": "text", "text": {"content": "(empty digest)"}}]}
        }]

    parts = []
    buf = md
    while buf:
        parts.append(buf[:max_chars_per_block])
        buf = buf[max_chars_per_block:]

    blocks = []
    for p in parts:
        blocks.append({
            "object": "block",
            "type": "code",
            "code": {
                "rich_text": [{"type": "text", "text": {"content": p}}],
                "language": "markdown"
            }
        })
    return blocks

def _divider_block() -> Dict[str, Any]:
    return {"object": "block", "type": "divider", "divider": {}}

def _heading_block(text: str) -> Dict[str, Any]:
    return {
        "object": "block",
        "type": "heading_2",
        "heading_2": {"rich_text": [{"type": "text", "text": {"content": text}}]}
    }

def _toggle_block(title: str, children: List[Dict[str, Any]]) -> Dict[str, Any]:
    return {
        "object": "block",
        "type": "toggle",
        "toggle": {
            "rich_text": [{"type": "text", "text": {"content": title}}],
            "children": children
        }
    }

notion_digest_page_id = None
notion_digest_page_url = None

if not NOTION_WEEKLY_DIGESTS_DS_ID:
    logger.warning("NOTION_WEEKLY_DIGESTS_DS_ID not set - skipping Notion writeback")
else:
    try:
        digest_title = f"Weekly Events Digest: {week_label}"

        logger.info(f"Upserting weekly digest to Notion data source: {NOTION_WEEKLY_DIGESTS_DS_ID}")
        logger.info(f"Digest title key: {digest_title}")

        # Search by title "contains" (safe, no dependency on custom properties)
        # NOTE: data_sources filter schema supports property filtering; title property key is unknown.
        # We will:
        #   - query first page of results without filter (small) is risky
        # Better approach:
        #   - attempt filter with common title keys ("Name", "Title"), then fallback to scanning.
        def query_by_title_key(title_key: str) -> List[Dict[str, Any]]:
            f = {"property": title_key, "title": {"contains": digest_title}}
            out = []
            cursor = None
            for _ in range(5):  # up to 5 pages
                res = query_data_source(NOTION_WEEKLY_DIGESTS_DS_ID, filter_obj=f, page_size=50, start_cursor=cursor)
                out.extend(res.get("results", []) or [])
                if not res.get("has_more"):
                    break
                cursor = res.get("next_cursor")
            return out

        candidates = []
        for key_guess in ["Name", "Title"]:
            try:
                found = query_by_title_key(key_guess)
                if found:
                    candidates = found
                    break
            except Exception:
                pass

        # Fallback: scan recent pages if we couldn't filter (last resort)
        if not candidates:
            logger.warning("Could not find digest via title filter. Falling back to scanning first 100 pages.")
            res = query_data_source(NOTION_WEEKLY_DIGESTS_DS_ID, page_size=100)
            candidates = res.get("results", []) or []

        # Pick the best match: exact title equals if possible, else contains
        existing_page = None
        for p in candidates:
            title_key = _find_title_property_key(p)
            t = ""
            try:
                t = prop_title(p.get("properties", {}) or {}, title_key)  # reuse Cell 05 helper if available
            except Exception:
                # local parse fallback
                props = p.get("properties", {}) or {}
                arr = ((props.get(title_key, {}) or {}).get("title", [])) if isinstance(props.get(title_key, {}), dict) else []
                if isinstance(arr, list):
                    t = "".join([str(x.get("plain_text","")) for x in arr if isinstance(x, dict)]).strip()
            if t == digest_title:
                existing_page = p
                break
        if not existing_page:
            for p in candidates:
                title_key = _find_title_property_key(p)
                # parse
                props = p.get("properties", {}) or {}
                arr = ((props.get(title_key, {}) or {}).get("title", [])) if isinstance(props.get(title_key, {}), dict) else []
                t = "".join([str(x.get("plain_text","")) for x in arr if isinstance(x, dict)]).strip() if isinstance(arr, list) else ""
                if digest_title in t:
                    existing_page = p
                    break

        # Build blocks: append-only + audit-friendly
        stamp = now_jst.strftime("%Y-%m-%d %H:%M:%S %Z")
        header_blocks = [
            _divider_block(),
            _heading_block(f"Digest Update ({stamp})"),
        ]
        digest_blocks = _markdown_to_blocks_minimal(weekly_digest_md)
        # Put digest into a toggle to avoid clutter
        toggle = _toggle_block(f"Weekly Digest Markdown — {week_label} (run_id={RUN_ID})", digest_blocks)
        blocks_to_add = header_blocks + [toggle]

        if existing_page:
            notion_digest_page_id = existing_page.get("id")
            # Detect actual title key for safe update
            title_key = _find_title_property_key(existing_page)
            logger.info(f"Found existing digest page: {notion_digest_page_id} (title_key='{title_key}') — updating title + appending blocks")

            # Update title only (safe regardless of DB schema)
            update_page(notion_digest_page_id, properties=_build_title_properties(title_key, digest_title))

            # Append blocks in chunks of 100
            for chunk in _chunk_blocks(blocks_to_add, 100):
                append_blocks(notion_digest_page_id, chunk)

        else:
            logger.info("No existing digest page found — creating a new one")

            # Create requires knowing title property name; most DBs use "Name"
            # We'll create with "Name" then correct key if needed by reading response.
            create_props = {"Name": _title_prop(digest_title)}
            new_page = create_page(NOTION_WEEKLY_DIGESTS_DS_ID, properties=create_props, children_blocks=None)
            notion_digest_page_id = new_page.get("id")

            # If title key isn't "Name", update accordingly
            title_key = _find_title_property_key(new_page)
            if title_key != "Name":
                update_page(notion_digest_page_id, properties=_build_title_properties(title_key, digest_title))

            for chunk in _chunk_blocks(blocks_to_add, 100):
                append_blocks(notion_digest_page_id, chunk)

        if notion_digest_page_id:
            notion_digest_page_url = f"https://www.notion.so/{notion_digest_page_id.replace('-', '')}"
            logger.info(f"Notion digest page URL: {notion_digest_page_url}")

    except Exception as e:
        logger.error(f"Notion writeback failed: {e}")
        logger.exception(e)
        notion_digest_page_id = None
        notion_digest_page_url = None

logger.info("Cell 10 complete: Notion writeback attempted")


2026-02-06 15:04:58,298 [WARNING] NOTION_WEEKLY_DIGESTS_DS_ID not set - skipping Notion writeback
2026-02-06 15:04:58,302 [INFO] Cell 10 complete: Notion writeback attempted


In [24]:
# ============================================================
# Cell 11 — Export local artifacts + final summary + on-screen preview (SCHEMA-ALIGNED REPLACEMENT)
# ============================================================
# Overview:
#   Export weekly digest markdown + JSON artifacts to output/ and show a readable on-screen preview:
#     - Top themes list
#     - A short "what changed" digest excerpt
#     - Notion page link (if written)
#
# Inputs / Outputs:
#   Inputs:  weekly_digest_md (str)
#            themes, top_themes, filtered_events, normalized_events, fetched_events
#            RUN_ID, now_jst, week_label, week_start_str, week_end_str, TIMEZONE
#            notion_digest_page_id, notion_digest_page_url (from Cell 10)
#   Outputs: Files in OUTPUT_DIR, prints + display preview
#
# Notes:
#   - Updates JSON schema keys to match new theme structure (theme_title, signals, etc.)
#   - Adds screen preview via IPython.display for quick inspection
#

import os
import json
from pathlib import Path
from datetime import datetime
from collections import defaultdict
from IPython.display import display, Markdown

# Ensure output dir
OUTPUT_DIR = Path(globals().get("OUTPUT_DIR", "output"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ----------------------------
# Export markdown
# ----------------------------
md_filename = f"weekly_events_digest_{week_start_str}.md"
md_filepath = OUTPUT_DIR / md_filename

try:
    md_filepath.write_text(weekly_digest_md, encoding="utf-8")
    logger.info(f"Exported markdown digest: {md_filepath}")
except Exception as e:
    logger.error(f"Failed to export markdown: {e}")

# ----------------------------
# Build JSON artifact (schema-aligned)
# ----------------------------
def _safe_float(x, default=0.0):
    try:
        return float(x)
    except Exception:
        return default

def _event_to_json(e: dict) -> dict:
    return {
        "notion_page_id": e.get("notion_page_id"),
        "title": e.get("title"),
        "detected_at": e.get("detected_at"),
        "event_date": e.get("event_date"),
        "event_type": e.get("event_type"),
        "source": e.get("source"),
        "source_url": e.get("source_url"),
        "summary_text": e.get("summary_text"),
        "confidence": _safe_float(e.get("confidence"), 0.0),
        "dedup_key": e.get("dedup_key"),
        "status": e.get("status"),
        "run_id": e.get("run_id"),
        "ingested_at": e.get("ingested_at"),
        "action_needed": bool(e.get("action_needed", False)),
        "target_ids": e.get("target_ids", []),
        "related_paper_ids": e.get("related_paper_ids", []),
    }

json_data = {
    "run_id": RUN_ID,
    "generated_at": now_jst.isoformat(),
    "week_label": week_label,
    "period": {
        "start": week_start_str,
        "end": week_end_str,
        "timezone": str(TIMEZONE),
    },
    "statistics": {
        "total_events_fetched": len(fetched_events) if "fetched_events" in globals() else None,
        "events_after_normalization": len(normalized_events),
        "events_after_filtering": len(filtered_events),
        "total_themes": len(themes),
        "top_themes_count": len(top_themes),
    },
    "top_themes": [
        {
            "rank": idx + 1,
            "theme_id": t.get("theme_id"),
            "theme_title": t.get("theme_title"),
            "theme_summary": t.get("theme_summary"),
            "why_it_matters": t.get("why_it_matters", []),
            "signals": t.get("signals", {}),
            "score": _safe_float(t.get("score"), 0.0),
            "score_breakdown": t.get("score_breakdown", {}),
            "event_count": int(t.get("event_count") or 0),
            "targets": t.get("targets", []),
            "types": t.get("types", []),
            "sources": t.get("sources", []),
            "keywords": t.get("keywords", []),
            "key_event_page_ids": t.get("key_event_page_ids", []),
            "top_events": [_event_to_json(e) for e in (t.get("top_events", []) or [])],
        }
        for idx, t in enumerate(top_themes)
    ],
    "notion_output": {
        "page_id": globals().get("notion_digest_page_id", None),
        "page_url": globals().get("notion_digest_page_url", None),
    },
}

json_filename = f"weekly_events_digest_{week_start_str}.json"
json_filepath = OUTPUT_DIR / json_filename

try:
    json_filepath.write_text(json.dumps(json_data, indent=2, ensure_ascii=False), encoding="utf-8")
    logger.info(f"Exported JSON artifact: {json_filepath}")
except Exception as e:
    logger.error(f"Failed to export JSON: {e}")

# ----------------------------
# Log final summary
# ----------------------------
logger.info("=" * 70)
logger.info("WEEKLY EVENTS DIGEST — RUN SUMMARY")
logger.info("=" * 70)
logger.info(f"Run ID: {RUN_ID}")
logger.info(f"Week: {week_label} ({week_start_str} to {week_end_str})")
logger.info(f"Timezone: {TIMEZONE}")
logger.info("")
logger.info("STATISTICS:")
logger.info(f"  Events normalized: {len(normalized_events)}")
logger.info(f"  Events after filtering: {len(filtered_events)}")
logger.info(f"  Total themes created: {len(themes)}")
logger.info(f"  Top themes selected: {len(top_themes)}")

# Event Type breakdown
type_counts = defaultdict(int)
for e in filtered_events:
    et = str(e.get("event_type","")).strip() or "UNKNOWN"
    type_counts[et] += 1
logger.info(f"  Event Type breakdown: {dict(sorted(type_counts.items(), key=lambda kv: kv[1], reverse=True))}")

logger.info("")
logger.info("TOP THEMES:")
for idx, t in enumerate(top_themes, 1):
    logger.info(f"  {idx}. {t.get('theme_title','(no title)')}")
    logger.info(f"     Score: {_safe_float(t.get('score'), 0.0):.2f} | Events: {t.get('event_count')} | Targets: {len(t.get('targets',[]))} | Signals: {t.get('signals',{})}")

logger.info("")
logger.info("OUTPUT ARTIFACTS:")
logger.info(f"  Markdown: {md_filepath}")
logger.info(f"  JSON: {json_filepath}")
if globals().get("notion_digest_page_url"):
    logger.info(f"  Notion page: {globals().get('notion_digest_page_url')}")
else:
    logger.info("  Notion page: (skipped or failed)")
logger.info("=" * 70)
logger.info("Weekly Events Digest run complete")
logger.info("=" * 70)

# ----------------------------
# On-screen preview (human readable)
# ----------------------------
preview_lines = []
preview_lines.append(f"# ✅ Weekly Events Digest Completed — {week_label}")
preview_lines.append("")
preview_lines.append(f"- **Run ID:** `{RUN_ID}`")
preview_lines.append(f"- **Period:** {week_start_str} → {week_end_str} ({TIMEZONE})")
preview_lines.append(f"- **Events:** {len(filtered_events)} (filtered) / {len(normalized_events)} (normalized)")
preview_lines.append(f"- **Themes:** {len(themes)} discovered, **{len(top_themes)}** selected")
if globals().get("notion_digest_page_url"):
    preview_lines.append(f"- **Notion:** {globals().get('notion_digest_page_url')}")
preview_lines.append("")
preview_lines.append("## Top themes (quick list)")
for i, t in enumerate(top_themes, 1):
    sig = t.get("signals", {}) or {}
    preview_lines.append(
        f"{i}. **{t.get('theme_title','(no title)')}** "
        f"(events={t.get('event_count')}, score={_safe_float(t.get('score'),0.0):.2f}, "
        f"momentum={sig.get('momentum','medium')})"
    )

preview_lines.append("")
preview_lines.append("## What changed (digest excerpt)")
# Take first ~80 lines of markdown to preview without flooding
md_excerpt = (weekly_digest_md or "").splitlines()
excerpt = "\n".join(md_excerpt[:80])
preview_lines.append("```markdown")
preview_lines.append(excerpt)
if len(md_excerpt) > 80:
    preview_lines.append("... (truncated) ...")
preview_lines.append("```")

preview_lines.append("")
preview_lines.append("## Files")
preview_lines.append(f"- Markdown: `{md_filepath.as_posix()}`")
preview_lines.append(f"- JSON: `{json_filepath.as_posix()}`")

display(Markdown("\n".join(preview_lines)))

logger.info("Cell 11 complete: Artifacts exported, summary logged, preview displayed")


2026-02-06 15:07:37,929 [INFO] Exported markdown digest: output/weekly_events_digest_2026-01-30.md
2026-02-06 15:07:37,957 [INFO] Exported JSON artifact: output/weekly_events_digest_2026-01-30.json
2026-02-06 15:07:37,958 [INFO] ======================================================================
2026-02-06 15:07:37,959 [INFO] WEEKLY EVENTS DIGEST — RUN SUMMARY
2026-02-06 15:07:37,961 [INFO] ======================================================================
2026-02-06 15:07:37,963 [INFO] Run ID: 20260206_134248
2026-02-06 15:07:37,964 [INFO] Week: 2026-01-30–2026-02-06 (2026-01-30 to 2026-02-06)
2026-02-06 15:07:37,966 [INFO] Timezone: Asia/Tokyo
2026-02-06 15:07:37,967 [INFO] 
2026-02-06 15:07:37,968 [INFO] STATISTICS:
2026-02-06 15:07:37,969 [INFO]   Events normalized: 72
2026-02-06 15:07:37,972 [INFO]   Events after filtering: 72
2026-02-06 15:07:37,974 [INFO]   Total themes created: 9
2026-02-06 15:07:37,977 [INFO]   Top themes selected: 5
2026-02-06 15:07:37,979 [INFO]   Eve

# ✅ Weekly Events Digest Completed — 2026-01-30–2026-02-06

- **Run ID:** `20260206_134248`
- **Period:** 2026-01-30 → 2026-02-06 (Asia/Tokyo)
- **Events:** 72 (filtered) / 72 (normalized)
- **Themes:** 9 discovered, **5** selected

## Top themes (quick list)
1. **Nvidia's Strategic Moves in AI Investment** (events=40, score=63.21, momentum=high)
2. **AI Leadership and Market Dynamics** (events=14, score=55.17, momentum=high)
3. **Advancements in UK Innovation and Policy** (events=7, score=45.56, momentum=high)
4. **Updates in Health Policy and Guidance** (events=3, score=40.25, momentum=high)
5. **VC Landscape Shifts Amid AI Focus** (events=2, score=34.99, momentum=high)

## What changed (digest excerpt)
```markdown
# Weekly Events Digest: 2026-01-30–2026-02-06

**Period:** 2026-01-30 to 2026-02-06
**Timezone:** Asia/Tokyo
**Generated:** 2026-02-06 13:43:00 JST
**Run ID:** 20260206_134248

---

## Weekly Coverage

- **Events processed (post-filter):** 72
- **Themes discovered:** 9
- **Top themes selected:** 5

### Event Type Breakdown

- PEOPLE: 54
- POLICY: 15
- VC: 3

---

## Executive Summary

- **1. Nvidia's Strategic Moves in AI Investment**  _(momentum=high, novelty=medium, uncertainty=medium)_
  - Nvidia CEO Jensen Huang has reaffirmed the company's commitment to invest in OpenAI, dismissing rumors of a stalled deal. This comes amid a broader context of declining software stocks, which Huang attributes to misconceptions about AI replacing existing tools…
- **2. AI Leadership and Market Dynamics**  _(momentum=high, novelty=medium, uncertainty=medium)_
  - This week, OpenAI's CEO Sam Altman made significant announcements regarding the company's direction and personnel changes, including the appointment of Dylan Scandrett as Head of Preparedness. Altman also addressed the hype surrounding the AI social network Mo…
- **3. Advancements in UK Innovation and Policy**  _(momentum=high, novelty=medium, uncertainty=medium)_
  - This week saw significant policy initiatives aimed at enhancing innovation in the UK. Notable developments include funding for net zero energy networks and advancements in quantum technology through UK-Japan partnerships. These changes signal a strong commitme…
- **4. Updates in Health Policy and Guidance**  _(momentum=high, novelty=medium, uncertainty=medium)_
  - This week saw significant updates in health policy, including new guidance for semaglutide prescribers and a public advisory against certain non-sterile alcohol-free wipes. These changes reflect a growing emphasis on patient safety and the need for individuali…
- **5. VC Landscape Shifts Amid AI Focus**  _(momentum=high, novelty=medium, uncertainty=medium)_
  - Peak XV has experienced internal disagreements leading to partner exits, prompting a strategic shift towards artificial intelligence. The firm is also transitioning board roles and expanding its presence in the U.S. while maintaining a strong focus on the Indi…

---

## Top 5 Themes

## 1. Nvidia's Strategic Moves in AI Investment

**Events:** 40  |  **Score:** 63.21
**Signals:** novelty=medium, momentum=high, uncertainty=medium
**Types:** PEOPLE
**Sources:** WEB
**Targets:** 2fb8e0e4-d162-81f9-aecc-f19a28fad58b
**Keywords:** nvidia, jensen, ceo, huang, openai, investment, company, billion, september, denies, partnership, chipmaker (+6)

### Theme Summary

Nvidia CEO Jensen Huang has reaffirmed the company's commitment to invest in OpenAI, dismissing rumors of a stalled deal. This comes amid a broader context of declining software stocks, which Huang attributes to misconceptions about AI replacing existing tools. The collaboration with OpenAI is positioned as a significant step for Nvidia as it navigates the evolving AI landscape.

### Why it matters

- Nvidia's investment in OpenAI could reshape the AI market and influence future tech developments.
- Huang's dismissal of AI replacement fears may stabilize investor confidence in software stocks.
- The partnership with OpenAI signals Nvidia's intent to maintain leadership in AI technology.

### Key Events

- **[CNBC Daily Open: Nvidia denies rift with OpenAI, while software and asset management stocks plunge](https://www.cnbc.com/2026/02/04/cnbc-daily-open-nvidia-denies-rift-with-openai-while-software-and-asset-management-stocks-plunge.html)** (PEOPLE | WEB) — 2026-02-06  (conf=0.60)
  Nvidia CEO Jensen Huang told CNBC's Jim Cramer on Tuesday that there's "no drama involved" between the company and OpenAI. "Everything's on track," he added.

- **[Nvidia nears deal to invest $20 billion in OpenAI funding round: Report](https://economictimes.indiatimes.com/tech/technology/update-1-nvidia-nears-deal-to-invest-20-billion-in-openai-funding-round-bloomberg-news-reports/articleshow/127898500.cms)** (PEOPLE | WEB) — 2026-02-06  (conf=0.60)
  Nvidia ‍CEO Jensen Huang ‍told CNBC earlier in the day ‌that the company would consider investing in OpenAI's next fundraising round and the startup's eventual IPO, following recent reports that the d…

- **[Jensen Huang clarifies collaboration with OpenAI on track, confirms participation in new funding round](https://www.digitimes.com/news/a20260204PD222/openai-nvidia-investment-funding-jensen-huang.html)** (PEOPLE | WEB) — 2026-02-06  (conf=0.60)
  Recent reports suggested a stall in investment between Nvidia and OpenAI, but Nvidia CEO Jensen Huang has confirmed that their collaboration remains on track. Huang stated in a CNBC interview that Nvi…

- **[Nvidia's Huang dismisses fears AI will replace software tools as stock selloff deepens](https://economictimes.indiatimes.com/markets/us-stocks/news/nvidias-huang-dismisses-fears-ai-will-replace-software-tools-as-stock-selloff-deepens/articleshow/127902319.cms)** (PEOPLE | WEB) — 2026-02-06  (conf=0.60)
  Nvidia CEO Jensen Huang dismissed fears that artificial intelligence ‍will replace software and related tools, calling the idea "illogical", after a significant ⁠selloff in global software stocks on T…

- **[CNBC Daily Open: UBS posts strong earnings while Novo Nordisk's U.S. shares crater on slowing growth](https://www.cnbc.com/2026/02/04/cnbc-daily-open-ubs-posts-strong-earnings-while-novo-nordisks-us-shares-crater-on-slowing-growth.html)** (PEOPLE | WEB) — 2026-02-06  (conf=0.60)
  Nvidia CEO Jensen Huang told CNBC's Jim Cramer on Tuesday that there's "no drama involved" between the company and OpenAI. "Everything's on track," he added.

- **[Nvidia is clamping down on certain 'T5T' emails that once circulated widely inside the company](https://www.businessinsider.com/nvidia-t5t-email-distribution-top-executives-2026-2)** (PEOPLE | WEB) — 2026-02-06  (conf=0.60)
  Nvidia has narrowed how certain 'Top 5 Things' emails are shared, a system long used to give CEO Jensen Huang insight into daily operations.

- **[Nvidia boss Jensen Huang says AI-replacement fears tanking software stocks is the 'most illogical thing in the world'](https://www.businessinsider.com/ai-software-tech-stocks-sell-off-nvidia-jensen-huang-illogical-2026-2)** (PEOPLE | WEB) — 2026-02-06  (conf=0.60)
... (truncated) ...
```

## Files
- Markdown: `output/weekly_events_digest_2026-01-30.md`
- JSON: `output/weekly_events_digest_2026-01-30.json`

2026-02-06 15:07:38,055 [INFO] Cell 11 complete: Artifacts exported, summary logged, preview displayed
